# V39 — Task suite (Nature/Nature Chemistry style) + strict LOCO + fusion + stacking/ensemble + task-design CV

核心改动（相对V21）：
1. **多视角融合（fused）**：将 raw / rel / log1p_rel_total 拼接。
2. **Top-K 集成读出**：每个外层LOCO折，在训练集内用GroupKFold选出Top-K配置，外层测试时做平均/投票。
3. **任务参数CV选择（design-CV）**：对 Notch-k / Double-Δ / Piecewise-w / BandPass-w 做分组CV选择（不使用测试折标签）。
4. **图形布局统一**：每个任务图均为“上：任务结果 / 下：准确性评估”，便于对照；图中不出现中文。

注意：为了平衡计算时间与准确性，本版本将模型搜索限制为“小模型库 + fused视角为主”，但保留严格的外层LOCO评价口径。

In [ ]:
import json

# ============================================================
# 0. 依赖导入与全局配置（中文注释尽量详细）
# ============================================================
import os, math, warnings, json
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import LeaveOneGroupOut, GroupKFold, StratifiedGroupKFold, GroupShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.linear_model import Ridge, LogisticRegression, ElasticNet, HuberRegressor, RidgeClassifier
from sklearn.svm import LinearSVC, SVC, SVR
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.cross_decomposition import PLSRegression
from sklearn.kernel_ridge import KernelRidge
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier

from sklearn.metrics import (
    r2_score, mean_absolute_error,
    matthews_corrcoef, balanced_accuracy_score,
    f1_score, confusion_matrix, roc_auc_score
)

from IPython.display import display, Image

warnings.filterwarnings("ignore")
np.set_printoptions(precision=4, suppress=True)

# -------------------------
# 输出目录与绘图分辨率
# -------------------------
OUTDIR = Path("V43_outputs_task_suite_universal_panels").resolve()
OUTDIR.mkdir(parents=True, exist_ok=True)

FIG_DPI = 300


# -------------------------
# Nature / Nature Chemistry 风格绘图（英文标签；更接近论文排版）
# -------------------------
def set_nature_style():
    """尽量模拟 Nature / Nature Chemistry 的简洁学术风格：
    - 白底
    - 去掉上/右边框
    - 轴线稍粗
    - 字体优先 Arial/Helvetica（若系统无则回退到 DejaVu Sans）
    注意：图中禁止出现中文，避免字体缺失警告。
    """
    plt.rcParams.update({
        "figure.facecolor": "white",
        "axes.facecolor": "white",
        "axes.spines.top": False,
        "axes.spines.right": False,
        "axes.linewidth": 1.2,
        "xtick.major.size": 3.5,
        "ytick.major.size": 3.5,
        "xtick.major.width": 1.1,
        "ytick.major.width": 1.1,
        "font.family": "sans-serif",
        "font.sans-serif": ["Arial", "Helvetica", "DejaVu Sans"],
        "axes.titlesize": 11,
        "axes.labelsize": 10,
        "xtick.labelsize": 9,
        "ytick.labelsize": 9,
        "legend.frameon": False,
        "legend.fontsize": 9,
        "lines.linewidth": 2.0,
    })



def beautify_ax(ax):
    """统一美化：轴向外刻度、细网格关闭、轻微留白。"""
    ax.tick_params(direction="out")
    ax.margins(x=0.02, y=0.08)

def add_panel_label(ax, label, x=-0.18, y=1.08):
    """在坐标轴左上角添加面板字母（a,b,c...），用于论文式排版。"""
    ax.text(x, y, label, transform=ax.transAxes,
            fontsize=12, fontweight="bold", va="top", ha="left")

set_nature_style()
print("OUTDIR =", OUTDIR)

# -------------------------
# 速度/准确性折中配置
# -------------------------
# 重要说明：
# - 数据量只有~32，若做“外层LOCO × 内层GroupKFold × 大网格 × 多视角”，计算会爆炸。
# - 因此V22采取：**fused 视角为主 + 小模型库 + Top-K集成**，常见能提升稳定性与分数。
# - 若你愿意更慢：可以把 SEARCH_FEATURE_MODES 扩展到 raw/rel 等；或增大网格。
PROFILE = dict(
    inner_folds=5,          # 内层分组交叉验证折数（会自动截断到训练集中可用的浓度水平数）
    topK_ensemble=5,        # Top-K 集成：更大的模型库下通常更稳（会更慢）
    degrees=[1, 2],         # 多项式扩展阶数（degree=2 会显著增加维数，但对Ridge/LogReg常有帮助）

    # ---------- 回归模型超参 ----------
    ridge_alpha=[0.003, 0.01, 0.03, 0.1, 0.3, 1, 3, 10, 30, 100],
    enet_alpha=[1e-4, 3e-4, 1e-3, 3e-3, 1e-2, 3e-2, 1e-1],
    enet_l1_ratio=[0.1, 0.3, 0.5, 0.7, 0.9],
    huber_alpha=[1e-4, 1e-3, 1e-2, 1e-1],
    huber_epsilon=[1.1, 1.2, 1.35, 1.5],
    pls_ncomp=[1, 2, 3, 4, 5, 6, 8],

    svr_C=[0.3, 1, 3, 10, 30, 100],
    svr_gamma=["scale", 0.01, 0.03, 0.1, 0.3, 1.0, 3.0],

    krr_alpha=[0.01, 0.1, 1, 10],
    krr_gamma=[0.01, 0.03, 0.1, 0.3, 1.0, 3.0],

    # 随机森林回归（小样本下容易过拟合，因此只给浅树；作为集成候选）
    rf_n_estimators=[300, 600],
    rf_max_depth=[2, 3, 4],
    rf_min_samples_leaf=[1, 2, 4],

    # ---------- 分类模型超参 ----------
    lin_C=[0.1, 0.3, 1, 3, 10, 30],
    ridgeclf_alpha=[0.1, 1, 10, 30, 100],
    rbf_C=[0.3, 1, 3, 10, 30, 100],
    svc_gamma=["scale", 0.01, 0.03, 0.1, 0.3, 1.0, 3.0],

    # ---------- 任务参数网格（仅用于 nested 任务参数选择时） ----------
    band_w=[3.0, 5.0, 7.0, 9.0],
    double_delta=[5.0, 7.5, 10.0, 12.5],
    piece_w=[8.0, 10.0, 12.0, 14.0],
    notch_k=[4.0, 6.0, 8.0, 10.0],

    # ---------- ranking / subset ----------
    rank_splits=180,
    rank_min_pairs=60,
    rank_max_pairs_train=4500,
    rank_max_pairs_test=2000,
    subset_repeats=90,
    subset_ks=[3, 5, 8, 12, 16, 20, 24, 28],

    # ---------- stacking / ensemble ----------
    meta_ridge_alpha=[0.01, 0.1, 1.0, 10.0, 100.0],
    meta_logreg_C=[0.1, 0.3, 1.0, 3.0, 10.0, 30.0],
    ensemble_tau=0.06,      # softmax温度：越小越“押宝”最强模型；小样本不宜过小

# ===== 稳定性/漂移控制（论文友好，优先启用） =====
# 回归任务默认使用“浓度代理（G_hat）→ 解析任务函数”的方式，通常更稳健：
# 1) 先在外层 LOCO 下得到浓度 OOF 预测 G_hat（并可用等距回归做校准）
# 2) 再用解析任务函数 y=f(G_hat) 生成 OOF 预测，从而显著提升回归任务的稳定性与准确性
reg_mode="proxy_analytic",  # 其它回归任务默认：proxy_analytic（更稳健，避免漂移）
reg_mode_doubletuning="proxy_analytic",  # DoubleTuning：Ĝ → 解析 DoubleTuning（推荐）
force_direct_doubletuning=True,  # 仅用于启用 DoubleTuning 的专用 reg_mode_doubletuning
prefer_g_proxy=True,                       # 是否训练浓度代理模型
x_proxy_calibrate_isotonic=True,  # Ĝ 校准：默认开启（小样本下等渗回归易压缩端点）

)


# -------------------------
# 特征视角：最终评估默认只搜索 fused（你也可以改成 ["fused","raw"] 增强，但更慢）
# -------------------------
FEATURE_MODES_ALL = ["raw", "log1p_raw", "rel", "log1p_rel_total", "snv_raw", "snv_rel", "clr_raw", "clr_rel", "fused"]
SEARCH_FEATURE_MODES = ["raw", "log1p_raw", "rel", "log1p_rel_total", "fused"]   
# 回归任务的候选特征模式：默认与 SEARCH_FEATURE_MODES 一致，避免依赖运行顺序
SEARCH_FEATURE_MODES_REG = list(SEARCH_FEATURE_MODES)
# 更慢但通常更准：在多视角上联合选参/集成   # 为了速度与稳定性：只在fused上做模型/超参搜索

# -------------------------
# 运行档位：用于控制“速度-准确性”折中（不改变评估口径：外层LOCO + 内层分组CV）
# -------------------------
运行档位 = "平衡"  # 可选："快速" / "平衡" / "冲高"
G0_stride = 1      # 扫描G0时的步长；>1 会显著加速（曲线更稀疏）

if 运行档位 == "快速":
    # 只用 fused 视角 + 线性读出为主；大幅缩小网格（通常可提速 5–15 倍）
    SEARCH_FEATURE_MODES = ["fused"]
    PROFILE.update(dict(
        inner_folds=3,
        topK_ensemble=3,
        degrees=[1],

        # 回归：以 Ridge/ElasticNet/Huber/PLS 为主，核方法只留少量候选
        ridge_alpha=[0.01, 0.1, 1, 10],
        enet_alpha=[1e-3, 1e-2, 1e-1],
        enet_l1_ratio=[0.2, 0.5, 0.8],
        huber_alpha=[1e-3, 1e-2],
        huber_epsilon=[1.2, 1.35],
        pls_ncomp=[1, 2, 3, 4],
        svr_C=[1, 10],
        svr_gamma=["scale", 0.1],
        krr_alpha=[], krr_gamma=[],  # 关闭KRR
        rf_n_estimators=[], rf_max_depth=[], rf_min_samples_leaf=[],  # 关闭RF（慢且小样本易过拟合）

        # 分类：仅线性模型 + LDA（稳定且快），关闭RBF-SVC
        lin_C=[0.3, 1, 3, 10],
        ridgeclf_alpha=[1, 10, 100],
        rbf_C=[], svc_gamma=[],

        # stacking：缩小元学习器网格
        meta_ridge_alpha=[0.1, 1.0, 10.0],
        meta_logreg_C=[0.3, 1.0, 3.0, 10.0],
        ensemble_tau=0.08,

        # ranking / subset：减少重复次数
        rank_splits=60,
        subset_repeats=25,
    ))
    G0_stride = 2

elif 运行档位 == "平衡":
    # 默认推荐：速度与分数通常比较均衡
    SEARCH_FEATURE_MODES = ["fused", "clr_rel", "snv_rel", "raw"]
    PROFILE.update(dict(
        inner_folds=4,
        topK_ensemble=5,
        degrees=[1],

        ridge_alpha=[0.01, 0.03, 0.1, 0.3, 1, 3, 10],
        enet_alpha=[3e-4, 1e-3, 3e-3, 1e-2, 3e-2],
        enet_l1_ratio=[0.2, 0.5, 0.8],
        huber_alpha=[1e-4, 1e-3, 1e-2],
        huber_epsilon=[1.2, 1.35, 1.5],
        pls_ncomp=[1, 2, 3, 4, 5],

        svr_C=[1, 3, 10],
        svr_gamma=["scale", 0.1, 1.0],
        krr_alpha=[0.1, 1, 10],
        krr_gamma=[0.1, 0.3, 1.0],
        rf_n_estimators=[200, 500], rf_max_depth=[3, 5, None], rf_min_samples_leaf=[1, 2, 4],  # 默认关RF

        lin_C=[0.3, 1, 3, 10, 30],
        ridgeclf_alpha=[0.1, 1, 10, 100],
        rbf_C=[0.3, 1.0, 3.0, 10.0], svc_gamma=["scale", 0.1, 1.0],  # 默认关RBF-SVC（慢且对小样本不稳）

        meta_ridge_alpha=[0.1, 1.0, 10.0],
        meta_logreg_C=[0.3, 1.0, 3.0, 10.0],
        ensemble_tau=0.06,

        rank_splits=120,
        subset_repeats=45,
    ))
    G0_stride = 1

else:
    # 冲高：保持V43默认（较慢，但更充分的网格与多视角）
    pass

print("运行档位 =", 运行档位)
print("SEARCH_FEATURE_MODES =", SEARCH_FEATURE_MODES)
print("inner_folds =", PROFILE["inner_folds"], "topK_ensemble =", PROFILE["topK_ensemble"], "degrees =", PROFILE["degrees"])

In [ ]:
# ============================================================
# 数据准备（V43）：优先读取“已处理峰面积特征CSV”，否则从 sample.zip / buffer.zip 自动重建
# 目标：让本笔记本在“只有原始txt + 两个zip”的情况下也能直接运行（无外部依赖）
# ============================================================

from pathlib import Path
import zipfile, re

# 若上游未定义CSV搜索模式，则在此给出默认值
if 'DATA_CSV_GLOB_LIST' not in globals():
    DATA_CSV_GLOB_LIST = [
        'X_peakArea_32x28_meanBlank*.csv',
        'X_peakArea_32x28_meanBlank.csv',
        'RC056.csv'
    ]


def _find_existing(paths):
    """按顺序返回第一个存在的路径（Path），否则返回None。"""
    for p in paths:
        if p is None:
            continue
        p = Path(p)
        if p.exists():
            return p
    return None

def _find_nearby(filename_list):
    """
    在以下位置查找文件：
    1) 当前工作目录
    2) /mnt/data（本环境挂载目录）
    """
    cwd = Path().resolve()
    for fn in filename_list:
        p = cwd / fn
        if p.exists():
            return p
    for fn in filename_list:
        p = Path("/mnt/data") / fn
        if p.exists():
            return p
    return None

def _read_chrom_ACh1_from_text(txt_lines):
    """
    从 Shimadzu LabSolutions 导出的txt中读取 Detector A-Ch1 的色谱（R.Time, Intensity）。
    返回：t(min), y(au) 的numpy数组
    """
    # 1) 定位数据段起始行（优先寻找表头 "R.Time\tIntensity"）
    start = None
    for i, line in enumerate(txt_lines):
        if line.strip() == "R.Time\tIntensity":
            start = i + 1
    if start is None:
        # 兜底：找第一个“数值\t数值”行
        num_pat = re.compile(r"^\s*\d+(\.\d+)?\t-?\d+(\.\d+)?\s*$")
        for i, line in enumerate(txt_lines):
            if num_pat.match(line):
                start = i
                break
    if start is None:
        raise ValueError("未在txt中找到色谱数据段（R.Time/Intensity）。")

    t, y = [], []
    for line in txt_lines[start:]:
        line = line.strip()
        if not line:
            continue
        parts = line.split("\t")
        if len(parts) < 2:
            continue
        try:
            tt = float(parts[0]); yy = float(parts[1])
        except Exception:
            continue
        t.append(tt); y.append(yy)

    t = np.asarray(t, float)
    y = np.asarray(y, float)
    if t.size < 10:
        raise ValueError("色谱数据点过少，疑似解析失败。")
    return t, y

def _extract_id_from_name(name):
    """从文件名中提取数字ID，用于排序与分组。"""
    stem = Path(name).stem
    m = re.search(r"(\d+)", stem)
    return int(m.group(1)) if m else 10**9

def _build_windows_from_peak_maxima(peaks_rt):
    """
    根据峰最大保留时间列表构造“互不重叠”的积分窗口：
    - 相邻峰的中点作为边界
    - 首尾窗口向外延伸半个相邻间距
    返回：[(start,end), ...] 长度=峰数
    """
    peaks_rt = np.asarray(peaks_rt, float)
    peaks_rt = np.sort(peaks_rt)
    mids = (peaks_rt[:-1] + peaks_rt[1:]) / 2.0
    left0 = peaks_rt[0] - (mids[0] - peaks_rt[0])
    rightN = peaks_rt[-1] + (peaks_rt[-1] - mids[-1])
    bounds = []
    for i in range(len(peaks_rt)):
        if i == 0:
            a, b = left0, mids[0]
        elif i == len(peaks_rt) - 1:
            a, b = mids[-1], rightN
        else:
            a, b = mids[i-1], mids[i]
        bounds.append((float(a), float(b)))
    return bounds

def _integrate_window(t, y, a, b, baseline_q=0.05):
    """
    在[a,b]窗口内积分（梯形积分）：
    - 先做一个稳健基线：减去窗口内的q分位数（默认5%）
    - 再把负值截断为0（避免噪声引入负面积）
    """
    mask = (t >= a) & (t <= b)
    if mask.sum() < 3:
        return 0.0
    yy = y[mask].astype(float)
    base = float(np.quantile(yy, baseline_q))
    yy = yy - base
    yy[yy < 0] = 0.0
    return float(np.trapz(yy, t[mask]))

def build_peakarea_dataframe_from_zips(sample_zip_path, buffer_zip_path, cache_csv_path=None):
    """
    从 sample.zip 与 buffer.zip 自动构建峰面积特征矩阵（32x28），并返回DataFrame：
    列：F1..F28, G
    说明：
    - G 采用 7.5 到 45，步长 2.5（共16个水平），每个水平两次重复（共32样品）
    - 该假设与本数据集的样品ID结构（两两成对且间隔固定）一致
    """
    # --- 峰最大保留时间：来自 Supporting Information 的 Table S3（约28个稳定峰） ---
    # 为了让脚本“自包含”，直接把峰位置写在这里，避免依赖外部 windows_28.csv
    peaks_rt = [
        9.6, 9.8, 10.0, 10.1, 10.3, 10.5, 10.6, 10.7, 11.0, 11.1, 11.3, 11.4, 11.5, 11.7,
        11.9, 12.2, 12.3, 12.4, 12.6, 12.9, 13.1, 13.3, 13.5, 13.8, 14.8, 14.9, 15.0, 15.7
    ]
    windows = _build_windows_from_peak_maxima(peaks_rt)

    # --- 读取buffer：用于计算“平均空白色谱” ---
    with zipfile.ZipFile(buffer_zip_path, "r") as zb:
        bnames = [n for n in zb.namelist() if n.lower().endswith(".txt")]
        if len(bnames) < 3:
            raise ValueError("buffer.zip中txt文件数量过少。")
        bnames = sorted(bnames, key=_extract_id_from_name)

        t_ref = None
        blank_stack = []
        for n in bnames:
            lines = zb.read(n).decode("utf-8", errors="ignore").splitlines()
            t, y = _read_chrom_ACh1_from_text(lines)
            if t_ref is None:
                t_ref = t
            else:
                # 若时间轴不完全一致，则线性插值到参考轴
                if (t.size != t_ref.size) or (not np.allclose(t, t_ref)):
                    y = np.interp(t_ref, t, y)
                    t = t_ref
            blank_stack.append(y)
        blank_mean = np.mean(np.vstack(blank_stack), axis=0)

    # --- 读取sample并提取峰面积 ---
    with zipfile.ZipFile(sample_zip_path, "r") as zs:
        snames = [n for n in zs.namelist() if n.lower().endswith(".txt")]
        if len(snames) != 32:
            raise ValueError(f"sample.zip中txt文件数量应为32，但当前为 {len(snames)}。请核对数据集。")
        snames = sorted(snames, key=_extract_id_from_name)

        # 16个浓度水平，每个2次重复
        G_levels = np.round(np.linspace(7.5, 45.0, 16), 6)
        G = np.repeat(G_levels, 2)
        if G.size != len(snames):
            raise ValueError("样品数与浓度映射不一致。")

        X = np.zeros((len(snames), len(windows)), float)
        for i, n in enumerate(snames):
            lines = zs.read(n).decode("utf-8", errors="ignore").splitlines()
            t, y = _read_chrom_ACh1_from_text(lines)
            if (t.size != t_ref.size) or (not np.allclose(t, t_ref)):
                y = np.interp(t_ref, t, y)
                t = t_ref

            y_corr = y - blank_mean  # 空白扣除
            for j, (a, b) in enumerate(windows):
                # 将窗口裁剪到时间范围内
                a2 = max(float(t[0]), a)
                b2 = min(float(t[-1]), b)
                X[i, j] = _integrate_window(t, y_corr, a2, b2, baseline_q=0.05)

        cols = [f"F{k}" for k in range(1, len(windows) + 1)]
        df = pd.DataFrame(X, columns=cols)
        df["G"] = G

    if cache_csv_path is not None:
        cache_csv_path = Path(cache_csv_path)
        cache_csv_path.parent.mkdir(parents=True, exist_ok=True)
        df.to_csv(cache_csv_path, index=False)

    return df


import glob

def find_data_csv():
    """
    按DATA_CSV_GLOB_LIST顺序查找已处理特征CSV。
    同时在当前目录与 /mnt/data 下搜索。
    """
    # 当前目录
    for pat in DATA_CSV_GLOB_LIST:
        hits = sorted(glob.glob(pat))
        if hits:
            return Path(hits[0])
    # /mnt/data
    md = Path("/mnt/data")
    for pat in DATA_CSV_GLOB_LIST:
        hits = sorted(glob.glob(str(md / pat)))
        if hits:
            return Path(hits[0])
    return None

# ---------- 1) 先尝试找已处理CSV ----------
data_csv = find_data_csv()

if data_csv is not None:
    print("Loading processed CSV:", data_csv)
    df = pd.read_csv(data_csv)
else:
    # ---------- 2) 找不到CSV则从zip重建 ----------
    sample_zip = _find_nearby(["sample.zip"])
    buffer_zip  = _find_nearby(["buffer.zip"])

    if sample_zip is None or buffer_zip is None:
        raise FileNotFoundError("未找到 sample.zip 或 buffer.zip。请把它们与本笔记本放在同一目录（或放到 /mnt/data）。")

    # 缓存输出：写成符合DATA_CSV_GLOB_LIST的命名，便于下次直接读取
    cache_csv = Path("X_peakArea_32x28_meanBlank_auto.csv")
    print("Rebuilding features from zips:", sample_zip, buffer_zip)
    df = build_peakarea_dataframe_from_zips(sample_zip, buffer_zip, cache_csv_path=cache_csv)
    print("Saved cache CSV:", cache_csv.resolve())

# ---------- 3) 统一列名并取出X与G ----------
peak_cols = [c for c in df.columns if re.fullmatch(r"F\d+", str(c).strip())]
peak_cols = sorted(peak_cols, key=lambda s: int(re.findall(r"\d+", s)[0]))

if len(peak_cols) < 8:
    raise ValueError(f"Peak feature columns too few: {len(peak_cols)}. Expecting F1..F28.")

# 优先使用列名为G的浓度列
if "G" in df.columns:
    g_col = "G"
else:
    cand = [c for c in df.columns if "glucose" in str(c).lower()]
    if cand:
        g_col = cand[0]
    else:
        nonF = [c for c in df.columns if c not in peak_cols]
        numeric_nonF = [c for c in nonF if np.issubdtype(df[c].dtype, np.number)]
        if not numeric_nonF:
            raise ValueError("无法推断浓度列，请将其命名为 'G'。")
        g_col = numeric_nonF[0]

X_raw = df[peak_cols].to_numpy(float)
G = df[g_col].to_numpy(float)

if not (np.all(np.isfinite(X_raw)) and np.all(np.isfinite(G))):
    raise ValueError("数据中存在NaN/Inf。")

n, p = X_raw.shape
G_min, G_max = float(G.min()), float(G.max())

# 将浓度线性映射到[0,1]（构造任务目标更方便）
x = (G - G_min) / (G_max - G_min + 1e-12)

# 外层分组：按“浓度水平”分组（避免同一浓度的重复测量同时出现在train/test导致泄漏）
groups = np.round(G, 6)
g_levels, g_counts = np.unique(groups, return_counts=True)

print("Shape X:", X_raw.shape, "G:", G.shape, "G col:", g_col)
print("Unique concentration levels:", len(g_levels), "replicates range:", (int(g_counts.min()), int(g_counts.max())))


In [ ]:
# ============================================================
# 2. 特征视角构造（含 fused 多视角拼接）
# ============================================================
def make_X_mode(X, mode: str):
    X = np.asarray(X, float)
    eps = 1e-12

    # 特殊模式：G_proxy（由“预测的归一化浓度”构造的低维代理特征）
    # 注意：X_GPROXY 必须在后续单元中先被构建（例如：先训练一个 LOCO 回归来预测 G_norm）。
    if mode == "G_proxy":
        if "X_GPROXY" not in globals():
            raise RuntimeError("未找到全局变量 X_GPROXY。请先运行 'G_proxy 构建' 单元。")
        return np.asarray(globals()["X_GPROXY"], float)

    if mode == "raw":
        return X

    if mode == "log1p_raw":
        return np.log1p(np.maximum(X, 0.0))

    if mode == "rel":
        s = X.sum(axis=1, keepdims=True)
        return X / (s + eps)

    if mode == "log1p_rel_total":
        s = X.sum(axis=1, keepdims=True)
        rel = X / (s + eps)
        z = np.log1p(np.maximum(rel, 0.0))
        total = np.log1p(np.maximum(s[:, 0], 0.0))[:, None]
        return np.concatenate([z, total], axis=1)

    # ===== 漂移鲁棒变换（论文友好，常见于色谱/光谱） =====
    if mode == "snv_raw":
        mu = np.mean(X, axis=1, keepdims=True)
        sd = np.std(X, axis=1, keepdims=True) + eps
        return (X - mu) / sd

    if mode == "snv_rel":
        s = X.sum(axis=1, keepdims=True)
        rel = X / (s + eps)
        mu = np.mean(rel, axis=1, keepdims=True)
        sd = np.std(rel, axis=1, keepdims=True) + eps
        return (rel - mu) / sd

    if mode == "clr_raw":
        Xp = np.clip(X, eps, None)
        L = np.log(Xp)
        return L - np.mean(L, axis=1, keepdims=True)

    if mode == "clr_rel":
        s = X.sum(axis=1, keepdims=True)
        rel = X / (s + eps)
        rel = np.clip(rel, eps, None)
        L = np.log(rel)
        return L - np.mean(L, axis=1, keepdims=True)

    if mode == "fused":
        A = make_X_mode(X, "raw")
        B = make_X_mode(X, "rel")
        C = make_X_mode(X, "log1p_rel_total")
        return np.concatenate([A, B, C], axis=1)

    raise ValueError(f"Unknown feature mode: {mode}")

# 自检：每种视角都应生成有限值
for m in FEATURE_MODES_ALL:
    Xm = make_X_mode(X_raw, m)
    assert Xm.shape[0] == n and np.all(np.isfinite(Xm))
print("Feature modes OK:", FEATURE_MODES_ALL)


In [ ]:

# ============================================================
# 3. 任务定义（新任务套件）
# ============================================================
def x_from_G(G_vec):
    """把葡萄糖浓度 G 映射到归一化坐标 x∈[0,1]。
    统一使用全局 G_min/G_max（来自数据），避免不同命名导致代理/目标不一致。
    """
    g = np.asarray(G_vec, float)
    return (g - float(G_min)) / (float(G_max) - float(G_min) + 1e-12)

def sigmoid(u):
    return 1.0 / (1.0 + np.exp(-u))

def tuning_target_from_x(x, x0, k):
    # 对称调谐：4*s*(1-s)，峰值=1，宽度由k控制
    s = sigmoid(k * (x - x0))
    return 4.0 * s * (1.0 - s)

# 回归任务族（输出[0,1]）
def notch_target(G0, k=6.0):
    x0 = (float(G0) - G_min) / (G_max - G_min + 1e-12)
    return 1.0 - tuning_target_from_x(x, x0, k)

def double_tuning_target(G0, delta_mM=7.5, k=6.0):
    x0 = (float(G0) - G_min) / (G_max - G_min + 1e-12)
    d = float(delta_mM) / (G_max - G_min + 1e-12)
    y = tuning_target_from_x(x, x0 - d, k) + tuning_target_from_x(x, x0 + d, k)
    mx = np.max(y)
    return (y / mx) if mx > 0 else y

def piecewise_saturation_target(G0, w_mM=10.0):
    x0 = (float(G0) - G_min) / (G_max - G_min + 1e-12)
    w = float(w_mM) / (G_max - G_min + 1e-12)
    y = (x - x0) / (w + 1e-12) + 0.5
    return np.clip(y, 0.0, 1.0)

# 分类任务（输出0/1）
def stripes_target(m=4):
    return (np.floor(m * x) % 2).astype(int)

def sine_sign_target(m=2):
    return (np.sin(2.0 * np.pi * m * x) > 0).astype(int)

def bandpass_target(G0, w_mM=5.0):
    return (np.abs(G - float(G0)) <= float(w_mM)).astype(int)

# 多级任务：K=4分箱 + 序数
def quantize_classes(K=4):
    edges = np.linspace(0.0, 1.0, K + 1)
    y = np.digitize(x, edges[1:-1], right=False).astype(int)
    return y, edges

def ordinal_targets_from_edges(edges):
    K = len(edges) - 1
    ys = []
    for t in range(1, K):
        ys.append((x >= edges[t]).astype(int))
    return ys


# ======================================================================
# 任务函数（支持在任意 G 向量上评估）：用于“G_proxy → 解析任务输出”的稳健回归
# ======================================================================
def _x_from_Gvals(Gvals):
    Gvals = np.asarray(Gvals, dtype=float)
    return (Gvals - G_min) / (G_max - G_min + 1e-12)

def notch_on_G(Gvals, G0, k=6.0):
    xh = _x_from_Gvals(Gvals)
    x0 = float((G0 - G_min) / (G_max - G_min + 1e-12))
    s = sigmoid(k * (xh - x0))
    y = 4.0 * s * (1.0 - s)
    return 1.0 - y

def double_tuning_on_G(G_vec, G0, delta_mM=7.5, k=6.0):
    """把“DoubleTuning”目标函数应用到任意的 G 向量（例如 G_hat）。
    重要：这里必须与 double_tuning_target(G0, ...) 的定义一致，避免目标/代理不匹配导致性能虚高或虚低。
    """
    x = x_from_G(np.asarray(G_vec, float))
    x0 = x_from_G(float(G0))
    d = float(delta_mM) / (G_max - G_min + 1e-12)

    y = tuning_target_from_x(x, x0 - d, k) + tuning_target_from_x(x, x0 + d, k)
    mx = float(np.max(y)) if np.size(y) else 0.0
    return (y / mx) if mx > 0 else y

def piecewise_sat_on_G(Gvals, G0, w_mM=10.0):
    xh = _x_from_Gvals(Gvals)
    x0 = float((G0 - G_min) / (G_max - G_min + 1e-12))
    w = float(w_mM / (G_max - G_min + 1e-12))
    # 0-1饱和的分段线性近似（与 piecewise_sat_target 一致）
    y = np.clip((xh - (x0 - 0.5*w)) / (w + 1e-12), 0.0, 1.0)
    return y

# G0扫描网格：用唯一浓度水平（去掉边缘点可降低边界效应）
G0_all = g_levels.copy()
G0_curve = G0_all.copy()
if len(G0_curve) > 6:
    G0_curve = G0_curve[1:-1]

# 根据运行档位对子采样，以显著降低“扫G0 × 嵌套CV”的总计算量
if "G0_stride" in globals():
    try:
        _st = int(G0_stride)
    except Exception:
        _st = 1
    if _st > 1 and len(G0_curve) > _st:
        G0_curve = G0_curve[::_st]
demo_G0 = float(np.median(G0_curve))

print("G0 points for curves:", len(G0_curve), "demo_G0 =", demo_G0)


In [ ]:
import inspect

# 兼容不同版本sklearn的LogisticRegression参数（例如某些环境不支持multi_class等）
def logreg_compat(**kwargs):
    from sklearn.linear_model import LogisticRegression as _LogisticRegression
    sig = inspect.signature(_LogisticRegression)
    allowed = set(sig.parameters.keys())
    filt = {k: v for k, v in kwargs.items() if k in allowed}
    return _LogisticRegression(**filt)


# ============================================================
# 4. Model Zoo（小模型库）与Pipeline构造
# ============================================================
# 说明：
# - 回归：Ridge / SVR(rbf) / KRR(rbf)
# - 二分类：LinearSVC / LogReg / SVC(rbf)
# - 多分类：LogReg / LinearSVC
# - 为了速度与稳定性：去掉了大规模RF/复杂网格


# 为了兼容 PLSRegression 的输出维度：
# - sklearn 的 PLSRegression.predict 往往返回 (n_samples, 1)
# - 但本任务的后续代码期望 1D 向量 (n_samples,)
# 因此用一个轻量包装，统一 ravel()。
class PLS1D(PLSRegression):
    def predict(self, X):
        y = super().predict(X)
        return np.asarray(y).ravel()


def make_reg_pipeline(name: str, degree: int = 1, **params):
    steps = [("scaler", StandardScaler())]
    if degree >= 2:
        steps += [("poly", PolynomialFeatures(degree=degree, include_bias=False)),
                  ("scaler2", StandardScaler())]

    if name == "Ridge":
        model = Ridge(alpha=float(params["alpha"]))
    elif name == "SVR_rbf":
        model = SVR(C=float(params["C"]), gamma=params.get("gamma", "scale"))
    elif name == "KRR_rbf":
        model = KernelRidge(alpha=float(params["alpha"]), kernel="rbf", gamma=float(params["gamma"]))

    elif name == "ElasticNet":
        model = ElasticNet(alpha=float(params["alpha"]), l1_ratio=float(params["l1_ratio"]), max_iter=20000)
    elif name == "Huber":
        model = HuberRegressor(alpha=float(params["alpha"]), epsilon=float(params["epsilon"]), max_iter=2000)
    elif name == "PLS":
        model = PLS1D(n_components=int(params["n_components"]))
    elif name == "RF":
        model = RandomForestRegressor(
            n_estimators=int(params["n_estimators"]),
            max_depth=int(params["max_depth"]),
            min_samples_leaf=int(params["min_samples_leaf"]),
            random_state=0,
        )

    else:
        raise ValueError(f"Unknown reg model: {name}")

    steps.append(("model", model))
    return Pipeline(steps)

def make_binclf_pipeline(name: str, degree: int = 1, **params):
    steps = [("scaler", StandardScaler())]
    if degree >= 2:
        steps += [
            ("poly", PolynomialFeatures(degree=degree, include_bias=False)),
            ("scaler2", StandardScaler()),
        ]

    # 小样本 + 类别不平衡：默认给出 balanced（对 band-pass 特别重要）
    if name == "LinearSVC":
        clf = LinearSVC(C=float(params["C"]), class_weight="balanced", max_iter=30000)
    elif name == "LogReg":
        clf = logreg_compat(
            C=float(params["C"]),
            max_iter=30000,
            solver=params.get("solver", "liblinear"),
            class_weight="balanced",
        )
    elif name == "SVC_rbf":
        # probability=True 会触发内部(非分组)交叉验证校准，既慢又不稳定；用 decision_function 更合适
        clf = SVC(
            C=float(params["C"]),
            gamma=params.get("gamma", "scale"),
            probability=False,
            class_weight="balanced",
        )

    elif name == "RidgeClf":
        clf = RidgeClassifier(alpha=float(params["alpha"]), class_weight="balanced")
    elif name == "LDA":
        # solver=lsqr 支持 shrinkage；对小样本、共线特征常更稳
        clf = LinearDiscriminantAnalysis(solver="lsqr", shrinkage=params.get("shrinkage", "auto"))

    else:
        raise ValueError(f"Unknown binary clf: {name}")

    steps.append(("model", clf))
    return Pipeline(steps)

def make_multiclf_pipeline(name: str, degree: int = 1, **params):
    steps = [("scaler", StandardScaler())]
    if degree >= 2:
        steps += [("poly", PolynomialFeatures(degree=degree, include_bias=False)),
                  ("scaler2", StandardScaler())]

    if name == "LogReg":
        clf = logreg_compat(C=float(params["C"]), max_iter=30000)
    elif name == "LinearSVC":
        clf = LinearSVC(C=float(params["C"]), max_iter=30000, class_weight="balanced")

    elif name == "LDA":
        clf = LinearDiscriminantAnalysis(solver="lsqr", shrinkage=params.get("shrinkage", "auto"))

    else:
        raise ValueError(f"Unknown multi clf: {name}")

    steps.append(("model", clf))
    return Pipeline(steps)


# -------------------------
# 超参网格（扩大：更慢但更可能找到更优配置）
# -------------------------
REG_GRID = []

# 说明：degree 只用于“线性读出 + 多项式特征”的模型（Ridge/ElasticNet/Huber/PLS）
#      对核方法/树模型，degree=1 通常足够，避免无意义的维数膨胀。
for deg in PROFILE["degrees"]:
    for a in PROFILE["ridge_alpha"]:
        REG_GRID.append(("Ridge", deg, dict(alpha=a)))
    for a in PROFILE["enet_alpha"]:
        for r in PROFILE["enet_l1_ratio"]:
            REG_GRID.append(("ElasticNet", deg, dict(alpha=a, l1_ratio=r)))
    for a in PROFILE["huber_alpha"]:
        for e in PROFILE["huber_epsilon"]:
            REG_GRID.append(("Huber", deg, dict(alpha=a, epsilon=e)))
    for nc in PROFILE["pls_ncomp"]:
        REG_GRID.append(("PLS", 1, dict(n_components=nc)))  # PLS 不做多项式扩展

# 核方法（默认 degree=1）
for C in PROFILE["svr_C"]:
    for g in PROFILE["svr_gamma"]:
        REG_GRID.append(("SVR_rbf", 1, dict(C=C, gamma=g)))
for a in PROFILE["krr_alpha"]:
    for g in PROFILE["krr_gamma"]:
        REG_GRID.append(("KRR_rbf", 1, dict(alpha=a, gamma=g)))

# 树模型（默认 degree=1；浅树作为集成候选）
for ne in PROFILE["rf_n_estimators"]:
    for md in PROFILE["rf_max_depth"]:
        for ms in PROFILE["rf_min_samples_leaf"]:
            REG_GRID.append(("RF", 1, dict(n_estimators=ne, max_depth=md, min_samples_leaf=ms)))


BINCLS_GRID = []
for deg in PROFILE["degrees"]:
    for C in PROFILE["lin_C"]:
        BINCLS_GRID.append(("LinearSVC", deg, dict(C=C)))
        BINCLS_GRID.append(("LogReg", deg, dict(C=C)))
    for a in PROFILE["ridgeclf_alpha"]:
        BINCLS_GRID.append(("RidgeClf", deg, dict(alpha=a)))
    # LDA 不用多项式
    BINCLS_GRID.append(("LDA", 1, dict(shrinkage="auto")))

for C in PROFILE["rbf_C"]:
    for g in PROFILE["svc_gamma"]:
        BINCLS_GRID.append(("SVC_rbf", 1, dict(C=C, gamma=g)))


# 多分类：模型候选网格（统一为三元组 (model_name, degree, params) ）
MULTICLS_GRID = []
for deg in PROFILE["degrees"]:
    for C in PROFILE["lin_C"]:
        MULTICLS_GRID.append(("LogReg", deg, dict(C=C)))
        MULTICLS_GRID.append(("LinearSVC", deg, dict(C=C)))

# RBF 核 SVC（degree无意义，用 1 占位）
for C in PROFILE.get("rbf_C", [1.0]):
    for g in PROFILE.get("svc_gamma", ["scale"]):
        MULTICLS_GRID.append(("SVC_rbf", 1, dict(C=C, gamma=g)))

# LDA（degree无意义，用 1 占位）
MULTICLS_GRID.append(("LDA", 1, dict(shrinkage="auto")))

# RandomForest（degree无意义，用 1 占位）
for ne in PROFILE.get("rf_n_estimators", []):
    for md in PROFILE.get("rf_max_depth", [None]):
        for ms in PROFILE.get("rf_min_samples_leaf", [1]):
            MULTICLS_GRID.append(("RF", 1, dict(n_estimators=ne, max_depth=md, min_samples_leaf=ms)))
print("Reg grid:", len(REG_GRID), "BinCls grid:", len(BINCLS_GRID), "MultiCls grid:", len(MULTICLS_GRID))

In [ ]:
# ============================================================
# 5. 指标与辅助函数（含Top-K集成） + 内层分组划分（StratifiedGroupKFold 优先）
# ============================================================
def phi_accuracy_from_mcc(mcc):
    return 0.5 * (float(mcc) + 1.0)

def regression_metrics(y_true, y_pred):
    return dict(
        r2=float(r2_score(y_true, y_pred)),
        mae=float(mean_absolute_error(y_true, y_pred)),
    )

def binary_metrics(y_true, y_pred):
    y_true = np.asarray(y_true, int)
    y_pred = np.asarray(y_pred, int)
    if len(np.unique(y_true)) < 2 or len(np.unique(y_pred)) < 2:
        mcc = 0.0
    else:
        mcc = float(matthews_corrcoef(y_true, y_pred))
    return dict(
        phi_acc=float(phi_accuracy_from_mcc(mcc)),
        bal_acc=float(balanced_accuracy_score(y_true, y_pred)),
    )

def multiclass_metrics(y_true, y_pred):
    return dict(
        macro_f1=float(f1_score(y_true, y_pred, average="macro")),
        bal_acc=float(balanced_accuracy_score(y_true, y_pred)),
    )

def softmax1d(z):
    z = np.asarray(z, float)
    z = z - np.max(z)
    e = np.exp(z)
    s = np.sum(e)
    return e / (s + 1e-12)

def predict_continuous_score(clf, X):
    """用于二分类/集成：统一成“连续得分”（越大越偏向1类）。
    - 优先 decision_function（更稳定、不需要内部概率校准）
    - 其次 predict_proba
    """
    if hasattr(clf, "decision_function"):
        s = clf.decision_function(X)
        return s if s.ndim == 1 else s[:, 0]
    if hasattr(clf, "predict_proba"):
        p = clf.predict_proba(X)
        if p.ndim == 2 and p.shape[1] >= 2:
            return p[:, 1]
    return clf.predict(X).astype(float)

def predict_multiclass_proba(clf, X):
    """多分类统一成“类概率矩阵”(n_samples × n_classes)。
    - LogisticRegression: predict_proba
    - LinearSVC: decision_function -> softmax 近似概率
    """
    if hasattr(clf, "predict_proba"):
        P = clf.predict_proba(X)
        return np.asarray(P, float)
    if hasattr(clf, "decision_function"):
        S = clf.decision_function(X)
        S = np.asarray(S, float)
        if S.ndim == 1:
            # 二分类退化成两列
            s = S
            P1 = 1.0 / (1.0 + np.exp(-s))
            return np.c_[1.0 - P1, P1]
        # 多分类：对每行softmax
        return np.vstack([softmax1d(row) for row in S])
    # 最保守：one-hot of predict
    y = clf.predict(X).astype(int)
    K = int(np.max(y)) + 1
    P = np.zeros((len(y), K), float)
    P[np.arange(len(y)), y] = 1.0
    return P

def groupkfold_inner_splits(train_idx, groups, n_splits, y=None, stratify=False, seed=0):
    """内层划分：
    - 默认 GroupKFold（只保证按浓度分组不泄漏）
    - 若 stratify=True 且提供 y：优先 StratifiedGroupKFold（减少“单类折”导致的无效评估）
    """
    train_idx = np.asarray(train_idx, int)
    gtr = groups[train_idx]
    n_groups = len(np.unique(gtr))
    ns = min(int(n_splits), int(n_groups))
    if ns < 2:
        return None

    if stratify and (y is not None):
        ytr = np.asarray(y)[train_idx]
        # 若ytr类别数不足，直接回退到GroupKFold
        if len(np.unique(ytr)) >= 2:
            try:
                sgkf = StratifiedGroupKFold(n_splits=ns, shuffle=True, random_state=int(seed))
                return list(sgkf.split(np.zeros(len(train_idx)), ytr, groups=gtr))
            except Exception:
                pass

    gkf = GroupKFold(n_splits=ns)
    return list(gkf.split(np.zeros(len(train_idx)), groups=gtr))

In [ ]:
# ============================================================
# 6. 嵌套评估核心（外层LOCO + 内层GroupKFold/StratifiedGroupKFold）
#    + Top-K集成 + Stacking（严格无泄漏）
# ============================================================

def softmax_weights(scores, tau=0.06):
    """把一组评分转换为softmax权重（评分越大越好）。"""
    s = np.asarray(scores, float)
    s = s - np.nanmax(s)
    w = np.exp(s / float(tau))
    w = w / (np.sum(w) + 1e-12)
    return w

def freeze_params(params):
    items = []
    for k, v in params.items():
        if hasattr(v, "item"):
            v = v.item()
        items.append((k, v))
    return tuple(sorted(items))

def thaw_params(frozen):
    return dict(frozen)

def best_threshold_by_mcc(y_true, scores, n_grid=121):
    y_true = np.asarray(y_true, int)
    s = np.asarray(scores, float)
    if len(s) < 6 or np.allclose(s, s[0]):
        return 0.0, 0.0
    qs = np.linspace(0.02, 0.98, int(n_grid))
    thr_list = np.unique(np.quantile(s, qs))
    best_mcc, best_thr = -1e18, 0.0
    for thr in thr_list:
        yp = (s >= thr).astype(int)
        if len(np.unique(yp)) < 2:
            continue
        mcc = matthews_corrcoef(y_true, yp)
        if mcc > best_mcc:
            best_mcc, best_thr = float(mcc), float(thr)
    if best_mcc < -1e17:
        return 0.0, 0.0
    return best_thr, best_mcc

# -------------------------
# 回归：单个外层fold（避免重复计算，支持“每fold任务参数不同”的nested模式）
# -------------------------
def regression_stack_onefold(
    tr, te, y, groups,
    X_modes, feature_modes, inner_folds, reg_grid,
    topK=3,
    meta_alpha_grid=(0.1, 1.0, 10.0),
    mix_grid=(0.0, 0.25, 0.5, 0.75, 1.0),
    tau=0.06,
    select_metric="mae",
    calibrate_isotonic=False,
):
    """回归 stacking（单个外层fold）。
    速度优化：对每个候选配置做“上界剪枝”
    - 若当前已完成的内层折平均分，即使剩余折都达到理论上界，也无法超过当前最优，则提前停止该配置。
    - 对 MAE（我们用 -MAE 当分数）上界为 0；对 R2 上界取 1。
    """
    if y is None:
        y = y_reg
    if y is None:
        raise ValueError('y 与 y_reg 不能同时为 None')
    y = np.asarray(y, float)
    inner = groupkfold_inner_splits(tr, groups, inner_folds)
    if inner is None:
        raise RuntimeError("Not enough groups for inner CV (reg).")

    # 1) inner：评估配置（带剪枝）
    scored = []
    best_so_far = -1e18
    F = len(inner)
    ub_max = 1.0 if select_metric == "r2" else 0.0  # 分数的理论上界

    for fm in feature_modes:
        Xf = X_modes[fm]
        for (model_name, deg, params) in reg_grid:
            ok = True
            ssum = 0.0
            kdone = 0
            for itr, ite in inner:
                tr2 = tr[itr]; te2 = tr[ite]
                try:
                    model = make_reg_pipeline(model_name, degree=deg, **params)
                    model.fit(Xf[tr2], y[tr2])
                    pred = model.predict(Xf[te2])
                    if select_metric == "r2":
                        sc = r2_score(y[te2], pred)
                    else:
                        sc = -mean_absolute_error(y[te2], pred)
                    if not np.isfinite(sc):
                        ok = False
                        break

                    ssum += float(sc)
                    kdone += 1

                    # 上界剪枝
                    ub = (ssum + float(ub_max) * (F - kdone)) / float(F)
                    if ub <= best_so_far - 1e-12:
                        ok = False
                        break
                except Exception:
                    ok = False
                    break

            if ok and kdone == F:
                mean_sc = float(ssum / float(F))
                scored.append((mean_sc, (fm, model_name, deg, params)))
                if mean_sc > best_so_far:
                    best_so_far = mean_sc

    if len(scored) == 0:
        pred_te = np.full(len(te), float(np.mean(y[tr])), float)
        cfg_rows = [dict(
            rank=1, feature_mode="NA", model="Mean", degree=-1,
            params="{}", inner_score=np.nan, meta_alpha=np.nan, mix=np.nan
        )]
        fold_r2 = regression_metrics(y[te], pred_te)["r2"]
        return pred_te, cfg_rows, float(fold_r2)

    scored.sort(key=lambda z: z[0], reverse=True)
    top = scored[:int(topK)]
    top_scores = [sc for sc, _ in top]
    w = softmax_weights(top_scores, tau=tau)

    # 2) 构造Z（n_tr × topK）
    Z = np.zeros((len(tr), len(top)), float)
    for j, (sc, (fm, model_name, deg, params)) in enumerate(top):
        Xf = X_modes[fm]
        z = np.full(len(tr), np.nan, float)
        for itr, ite in inner:
            tr2 = tr[itr]; te2 = tr[ite]
            model = make_reg_pipeline(model_name, degree=deg, **params)
            model.fit(Xf[tr2], y[tr2])
            z[ite] = model.predict(Xf[te2])
        if not np.all(np.isfinite(z)):
            model = make_reg_pipeline(model_name, degree=deg, **params)
            model.fit(Xf[tr], y[tr])
            z[:] = model.predict(Xf[tr])
        Z[:, j] = z

    # 3) 选meta alpha + mix
    best_a, best_mix, best_sc = None, None, -1e18
    for a in meta_alpha_grid:
        for mix in mix_grid:
            scs = []
            for itr, ite in inner:
                meta = Ridge(alpha=float(a))
                meta.fit(Z[itr], y[tr][itr])
                p_meta = meta.predict(Z[ite])
                p_avg = np.dot(Z[ite], w)
                p = (1.0 - float(mix)) * p_avg + float(mix) * p_meta
                p = np.clip(p, 0.0, 1.0)
                if select_metric == "r2":
                    scs.append(r2_score(y[tr][ite], p))
                else:
                    scs.append(-mean_absolute_error(y[tr][ite], p))
            sc = float(np.mean(scs)) if len(scs) else -1e18
            if sc > best_sc:
                best_sc, best_a, best_mix = sc, float(a), float(mix)
    if best_a is None:
        best_a, best_mix = 1.0, 1.0

    meta = Ridge(alpha=float(best_a)).fit(Z, y[tr])

    # 4) 测试：基模型重训得到Z_te
    Z_te = np.zeros((len(te), len(top)), float)
    cfg_rows = []
    for j, (sc, (fm, model_name, deg, params)) in enumerate(top, start=1):
        Xf = X_modes[fm]
        base = make_reg_pipeline(model_name, degree=deg, **params)
        base.fit(Xf[tr], y[tr])
        Z_te[:, j-1] = base.predict(Xf[te])
        cfg_rows.append(dict(
            rank=j, feature_mode=fm, model=model_name, degree=deg,
            params=str(params), inner_score=float(sc),
            meta_alpha=float(best_a), mix=float(best_mix)
        ))

    p_avg = np.dot(Z_te, w)
    p_meta = meta.predict(Z_te)
    pred_te = (1.0 - float(best_mix)) * p_avg + float(best_mix) * p_meta
    pred_te = np.clip(pred_te, 0.0, 1.0)

    # 可选：等渗回归（isotonic）校准，用于把预测映射到更符合 y 的单调标定
    # 这里用外层训练集的“inner OOF 预测”(Z -> pred_tr_uncal) 来拟合校准器，避免把测试组信息引入校准
    if calibrate_isotonic:
        try:
            from sklearn.isotonic import IsotonicRegression
            y_true_tr = y[tr]
            pred_tr_uncal = (1.0 - best_mix) * (Z @ w) + best_mix * meta.predict(Z)
            pred_tr_uncal = np.clip(pred_tr_uncal, 0.0, 1.0)
            iso = IsotonicRegression(out_of_bounds="clip")
            iso.fit(pred_tr_uncal, y_true_tr)
            pred_te = iso.transform(pred_te)
            pred_te = np.clip(pred_te, 0.0, 1.0)
            cfg_rows[0]["isotonic"] = True
        except Exception:
            cfg_rows[0]["isotonic"] = False

    fold_r2 = regression_metrics(y[te], pred_te)["r2"]
    return pred_te, cfg_rows, float(fold_r2)



# -------------------------
# 二分类：单个外层fold（严格无泄漏 stacking）
# -------------------------
def binary_stack_onefold(
    tr, te, y_bin, groups,
    X_modes, feature_modes, inner_folds, cls_grid,
    topK=3,
    meta_C_grid=(0.3, 1.0, 3.0, 10.0),
    tau=0.06,
    seed=0,
):
    y_bin = np.asarray(y_bin, int)
    y_true_te = y_bin[te]

    # 退化情形1：训练折只有单一类别 -> 直接输出多数类（避免模型训练/阈值优化报错）
    if len(np.unique(y_bin[tr])) < 2:
        maj = int(np.round(y_bin[tr].mean()))
        pred_te = np.full(len(te), maj, int)
        cfg_rows = [dict(
            rank=1, feature_mode="NA", model="Majority", degree=-1,
            params="{}", inner_score=np.nan, meta_C=np.nan, thr=np.nan
        )]
        met = binary_metrics(y_true_te, pred_te)
        fold_metrics = dict(
            phi_acc=float(met["phi_acc"]),
            acc=float(np.mean(y_true_te == pred_te)),
            bal_acc=float(met["bal_acc"]),
        )
        return pred_te, cfg_rows, fold_metrics

    # 内层分组CV（带分层：保证每个inner折尽量两类都有）
    inner = groupkfold_inner_splits(tr, groups, inner_folds, y=y_bin, stratify=True, seed=seed)

    # 退化情形2：组数不足导致无法做inner-CV -> 回退到多数类预测
    if inner is None:
        maj = int(np.round(y_bin[tr].mean()))
        pred_te = np.full(len(te), maj, int)
        cfg_rows = [dict(
            rank=1, feature_mode="NA", model="Majority", degree=-1,
            params="{}", inner_score=np.nan, meta_C=np.nan, thr=np.nan
        )]
        met = binary_metrics(y_true_te, pred_te)
        fold_metrics = dict(
            phi_acc=float(met["phi_acc"]),
            acc=float(np.mean(y_true_te == pred_te)),
            bal_acc=float(met["bal_acc"]),
        )
        return pred_te, cfg_rows, fold_metrics

    # ------------------------------------------------------------
    # 1) 内层CV：对（特征模式 × 模型族 × 超参）打分，挑top-K
    # ------------------------------------------------------------
    scored = []
    cache_inner = {}
    for fm in feature_modes:
        Xf = X_modes[fm]
        for (model_name, deg, params) in cls_grid:
            s_oof = np.full(len(tr), np.nan, float)
            ok = True
            for itr, ite in inner:
                tr2 = tr[itr]; te2 = tr[ite]
                if len(np.unique(y_bin[tr2])) < 2:
                    # inner折训练端退化：用训练均值当作概率
                    s_oof[ite] = float(np.mean(y_bin[tr2]))
                    continue
                try:
                    clf = make_binclf_pipeline(model_name, degree=deg, **params)
                    clf.fit(Xf[tr2], y_bin[tr2])
                    s_oof[ite] = predict_continuous_score(clf, Xf[te2])
                except Exception:
                    ok = False
                    break
            if ok and np.isfinite(s_oof).all():
                # 用φ-accuracy为主指标（同文稿定义），并记录bal_acc辅助
                met = binary_metrics(y_bin[tr], (s_oof >= 0.5).astype(int))
                score = float(met["phi_acc"])
                scored.append((score, fm, model_name, deg, params, float(met["bal_acc"])))
                cache_inner[(fm, model_name, deg, json.dumps(params, sort_keys=True))] = s_oof

    # 退化情形3：没有任何候选模型通过inner评估 -> 回退多数类
    if len(scored) == 0:
        maj = int(np.round(y_bin[tr].mean()))
        pred_te = np.full(len(te), maj, int)
        cfg_rows = [dict(
            rank=1, feature_mode="NA", model="Majority", degree=-1,
            params="{}", inner_score=np.nan, meta_C=np.nan, thr=np.nan
        )]
        met = binary_metrics(y_true_te, pred_te)
        fold_metrics = dict(
            phi_acc=float(met["phi_acc"]),
            acc=float(np.mean(y_true_te == pred_te)),
            bal_acc=float(met["bal_acc"]),
        )
        return pred_te, cfg_rows, fold_metrics

    scored.sort(key=lambda z: z[0], reverse=True)
    chosen = scored[:max(1, int(topK))]

    # ------------------------------------------------------------
    # 2) 外层训练：训练top-K基模型，得到te端连续分数（prob/score）
    # ------------------------------------------------------------
    base_scores_te = []
    cfg_rows = []
    for rank, (score, fm, model_name, deg, params, bal) in enumerate(chosen, start=1):
        Xf = X_modes[fm]
        key = (fm, model_name, deg, json.dumps(params, sort_keys=True))
        s_tr = cache_inner.get(key, None)

        # 外层在全部tr上重训基模型
        clf = make_binclf_pipeline(model_name, degree=deg, **params)
        clf.fit(Xf[tr], y_bin[tr])
        s_te = predict_continuous_score(clf, Xf[te])
        base_scores_te.append(s_te)

        cfg_rows.append(dict(
            rank=int(rank),
            feature_mode=str(fm),
            model=str(model_name),
            degree=int(deg),
            params=json.dumps(params, sort_keys=True),
            inner_score=float(score),
            meta_C=np.nan,
            thr=np.nan
        ))

    # ------------------------------------------------------------
    # 3) 软加权集成（Boltzmann权重）：用τ控制“接近硬top-1”还是“更平均”
    # ------------------------------------------------------------
    # 注：τ越小越接近只用第一名；τ越大越接近平均
    scores_arr = np.array([c[0] for c in chosen], float)
    if np.all(np.isfinite(scores_arr)) and len(scores_arr) > 0:
        scores_arr = scores_arr - scores_arr.max()
        w = np.exp(scores_arr / max(1e-6, float(tau)))
        w = w / (w.sum() + 1e-12)
    else:
        w = np.ones(len(base_scores_te), float) / max(1, len(base_scores_te))

    s_te_ens = np.zeros(len(te), float)
    for wi, s_te in zip(w, base_scores_te):
        s_te_ens += wi * s_te

    # ------------------------------------------------------------
    # ------------------------------------------------------------
    # 4) 阈值优化：在tr上用“加权OOF连续分数”选阈值（优先），缺失时回退到top-1 OOF
    # ------------------------------------------------------------
    # 说明：阈值学习只使用外层训练集tr的inner-OOF分数，不接触外层测试te，不会泄漏。
    base_scores_tr = []
    ok_tr = True
    for (score, fm, model_name, deg, params, bal) in chosen:
        key = (fm, model_name, deg, json.dumps(params, sort_keys=True))
        s_tr = cache_inner.get(key, None)
        if s_tr is None or (not np.isfinite(s_tr).all()):
            ok_tr = False
            break
        base_scores_tr.append(np.asarray(s_tr, float))

    if ok_tr and len(base_scores_tr) == len(base_scores_te):
        s_tr_ens = np.zeros(len(tr), float)
        for wi, s_tr in zip(w, base_scores_tr):
            s_tr_ens += wi * s_tr
        _thr = best_threshold_by_mcc(y_bin[tr], s_tr_ens)
        thr = float(_thr[0]) if isinstance(_thr, tuple) else float(_thr)
    else:
        # 回退：用top-1的OOF连续分数选阈值；若仍不可用，则用默认阈值（概率用0.5，间隔分数用0.0）
        best_fm, best_model, best_deg, best_params = chosen[0][1], chosen[0][2], chosen[0][3], chosen[0][4]
        key0 = (best_fm, best_model, best_deg, json.dumps(best_params, sort_keys=True))
        s_tr0 = cache_inner.get(key0, None)
        if s_tr0 is None or (not np.isfinite(s_tr0).all()):
            # 根据te端分数范围粗判是“概率”还是“决策间隔分数”
            mn, mx = float(np.nanmin(s_te_ens)), float(np.nanmax(s_te_ens))
            thr = 0.0 if (mn < -1e-6 or mx > 1.0 + 1e-6) else 0.5
        else:
            _thr = best_threshold_by_mcc(y_bin[tr], s_tr0)
            thr = float(_thr[0]) if isinstance(_thr, tuple) else float(_thr)
    pred_te = (s_te_ens >= thr).astype(int)

    # 把阈值写回第一名模型配置，便于后续汇总
    cfg_rows[0]["thr"] = float(thr)

    met = binary_metrics(y_true_te, pred_te)
    fold_metrics = dict(
        phi_acc=float(met["phi_acc"]),
        acc=float(np.mean(y_true_te == pred_te)),
        bal_acc=float(met["bal_acc"]),
    )
    return pred_te, cfg_rows, fold_metrics


def multiclass_stack_onefold(
    tr, te, y_mc, groups,
    X_modes, feature_modes, inner_folds, multi_grid,
    topK=3,
    meta_C_grid=(0.3, 1.0, 3.0, 10.0),
    tau=0.06,
    seed=0,
):
    """多分类 stacking（单个外层fold；严格无泄漏）。

    速度优化要点：
    1) 对每个候选配置，仅做“一遍”内层循环：
       - 同时产生 OOF 概率 P_oof（用于后续 stacking）
       - 同时由 argmax(P) 得到预测并计算 macro-F1（用于筛选配置）
       这样避免“先算F1、再算P_oof”的双遍训练。
    2) 上界剪枝：macro-F1 的理论上界为 1。
       若当前已完成折的平均分在“剩余折全为1”的情况下仍无法超过当前最优，则提前停止该配置评估。
    """
    y_mc = np.asarray(y_mc, int)

    inner = groupkfold_inner_splits(tr, groups, inner_folds, y=y_mc, stratify=True, seed=seed)
    if inner is None:
        raise RuntimeError("Not enough groups for inner CV (multi).")

    n_inner = len(inner)
    scored = []
    cache_oofP = {}
    best_mean_f1 = -1e18  # 用于剪枝

    for fm in feature_modes:
        Xf = X_modes[fm]
        for _item in multi_grid:
            # multi_grid 允许三种输入：
            #  1) (model_name, degree, params)
            #  2) (model_name, params) -> degree=1
            #  3) dict(model=..., degree=..., params=...)
            if isinstance(_item, dict):
                model_name = _item.get("model", _item.get("name"))
                deg = int(_item.get("degree", 1))
                params = _item.get("params", {})
            else:
                if len(_item) == 3:
                    model_name, deg, params = _item
                elif len(_item) == 2:
                    model_name, params = _item
                    deg = 1
                else:
                    raise ValueError(f"Invalid multi_grid item: {_item!r}")


        
            ok = True

            # 预先分配 OOF 概率矩阵（仅针对外层训练集 tr）
            Kc = int(np.max(y_mc[tr])) + 1
            P_oof = np.full((len(tr), Kc), np.nan, float)

            sum_f1 = 0.0
            done = 0

            for itr, ite in inner:
                tr2 = tr[itr]
                te2 = tr[ite]
                done += 1

                try:
                    clf = make_multiclf_pipeline(model_name, degree=deg, **params)
                    clf.fit(Xf[tr2], y_mc[tr2])

                    # 预测概率（用于 stacking）与预测类别（用于打分）一次性完成
                    P = predict_multiclass_proba(clf, Xf[te2])
                    if (P is None) or (not np.all(np.isfinite(P))):
                        ok = False
                        break

                    P_oof[ite, :] = P
                    pred = np.argmax(P, axis=1).astype(int)

                    f1 = f1_score(y_mc[te2], pred, average="macro")
                    if not np.isfinite(f1):
                        ok = False
                        break

                    sum_f1 += float(f1)

                    # 上界剪枝：剩余折即使全为1，也无法超过当前最优则提前停止
                    remain = n_inner - done
                    mean_upper = (sum_f1 + remain * 1.0) / n_inner
                    if mean_upper <= best_mean_f1 + 1e-12:
                        ok = False
                        break

                except Exception:
                    ok = False
                    break

            if (not ok) or (not np.all(np.isfinite(P_oof))):
                continue

            mean_f1 = sum_f1 / n_inner
            cfg_key = (fm, model_name, deg, freeze_params(params))
            cache_oofP[cfg_key] = P_oof
            scored.append((float(mean_f1), cfg_key))

            if mean_f1 > best_mean_f1:
                best_mean_f1 = float(mean_f1)

    # 若全部配置失败，退化为“多数类”
    if len(scored) == 0:
        maj = int(pd.Series(y_mc[tr]).mode().iloc[0])
        pred_te = np.full(len(te), maj, int)
        cfg_rows = [dict(
            rank=1, feature_mode="NA", model="Majority", degree=-1,
            params="{}", inner_score=np.nan, meta_C=np.nan
        )]
        fold_f1 = multiclass_metrics(y_mc[te], pred_te)["macro_f1"]
        return pred_te, cfg_rows, float(fold_f1)

    # 选 topK 配置并进行 stacking
    scored.sort(key=lambda z: z[0], reverse=True)
    top = scored[:int(topK)]
    top_scores = [sc for sc, _ in top]
    w = softmax_weights(top_scores, tau=tau)

    Kc = int(np.max(y_mc[tr])) + 1
    Z = np.zeros((len(tr), len(top) * Kc), float)
    for j, (sc, cfg_key) in enumerate(top):
        P = cache_oofP[cfg_key]
        Z[:, j*Kc:(j+1)*Kc] = P

    # 选择 meta 的正则强度（内层CV，仅用外层训练集）
    best_C, best_f1 = None, -1e18
    for C in meta_C_grid:
        f1s = []
        ok = True
        for itr, ite in inner:
            try:
                meta = logreg_compat(C=float(C), max_iter=30000, solver="lbfgs", class_weight="balanced")
                meta.fit(Z[itr], y_mc[tr][itr])
                pred = meta.predict(Z[ite]).astype(int)
                f1s.append(f1_score(y_mc[tr][ite], pred, average="macro"))
            except Exception:
                ok = False
                break
        if (not ok) or (not len(f1s)) or (not np.isfinite(np.mean(f1s))):
            continue
        m = float(np.mean(f1s))
        if m > best_f1:
            best_f1, best_C = m, float(C)

    if best_C is None:
        best_C = float(meta_C_grid[0])

    meta = logreg_compat(C=float(best_C), max_iter=30000, solver="lbfgs")
    meta.fit(Z, y_mc[tr])

    # 外层测试：生成各基学习器概率并喂给 meta
    Zte = np.zeros((len(te), len(top) * Kc), float)
    cfg_rows = []
    for j, (sc, cfg_key) in enumerate(top):
        fm, model_name, deg, frozen = cfg_key
        params = thaw_params(frozen)
        Xf = X_modes[fm]

        clf = make_multiclf_pipeline(model_name, degree=deg, **params)
        clf.fit(Xf[tr], y_mc[tr])

        Pte = predict_multiclass_proba(clf, Xf[te])
        Zte[:, j*Kc:(j+1)*Kc] = Pte

        cfg_rows.append(dict(
            rank=int(j+1),
            feature_mode=str(fm),
            model=str(model_name),
            degree=int(deg),
            params=str(params),
            inner_score=float(sc),
            meta_C=float(best_C),
            weight=float(w[j]),
        ))

    pred_te = meta.predict(Zte).astype(int)
    fold_f1 = multiclass_metrics(y_mc[te], pred_te)["macro_f1"]
    return pred_te, cfg_rows, float(fold_f1)

def nested_loco_regression_stack(
    y=None, y_reg=None, groups=None,
    feature_modes=None, inner_folds=5,
    reg_grid=None, topK=3,
    meta_alpha_grid=(0.1, 1.0, 10.0),
    mix_grid=(0.0, 0.25, 0.5, 0.75, 1.0),
    tau=0.06,
    select_metric="mae",
    calibrate_isotonic=False,
):
    # 兼容旧参数名：允许用 y_reg 传入回归标签
    if y is None:
        y = y_reg
    
    y = np.asarray(y, float)
    logo = LeaveOneGroupOut()
    X_modes = {fm: make_X_mode(X_raw, fm) for fm in feature_modes}

    oof = np.full_like(y, np.nan, dtype=float)
    fold_scores = []
    chosen_rows = []

    for fold_id, (tr, te) in enumerate(logo.split(np.zeros_like(y), groups=groups)):
        pred_te, cfg_rows, sc = regression_stack_onefold(
            tr, te, y, groups,
            X_modes, feature_modes, inner_folds, reg_grid,
            topK=topK, meta_alpha_grid=meta_alpha_grid, mix_grid=mix_grid,
            tau=tau, select_metric=select_metric, calibrate_isotonic=calibrate_isotonic
    )
        oof[te] = pred_te
        fold_scores.append(float(sc))
        for row in cfg_rows:
            row = dict(row)
            row["fold"] = int(fold_id)
            chosen_rows.append(row)

    assert np.all(np.isfinite(oof))
    return oof, pd.DataFrame(chosen_rows), np.array(fold_scores, float)


def nested_loco_binary_stack(
    y_bin, groups, feature_modes, inner_folds, cls_grid, topK=3,
    meta_C_grid=(0.3, 1.0, 3.0, 10.0),
    tau=0.06,
    seed=0,
):
    y_bin = np.asarray(y_bin, int)
    logo = LeaveOneGroupOut()
    X_modes = {fm: make_X_mode(X_raw, fm) for fm in feature_modes}

    oof = np.full_like(y_bin, -1, dtype=int)
    fold_rows = []
    chosen_rows = []

    for fold_id, (tr, te) in enumerate(logo.split(np.zeros_like(y_bin), groups=groups)):
        pred_te, cfg_rows, fold_m = binary_stack_onefold(
            tr, te, y_bin, groups,
            X_modes, feature_modes, inner_folds, cls_grid,
            topK=topK, meta_C_grid=meta_C_grid, tau=tau, seed=seed+fold_id
        )
        oof[te] = pred_te
        group_val = groups[te][0] if hasattr(groups, "__len__") else None
        fold_rows.append(dict(fold=int(fold_id), group=group_val, **fold_m))
        for row in cfg_rows:
            row = dict(row)
            row["fold"] = int(fold_id)
            chosen_rows.append(row)

    if np.any(oof < 0):
        raise RuntimeError("Invalid OOF predictions (binary).")
    return oof, pd.DataFrame(chosen_rows), pd.DataFrame(fold_rows)

def nested_loco_multiclass(
    y_mc, groups, feature_modes, inner_folds, multi_grid, topK=3,
    meta_C_grid=(0.3, 1.0, 3.0, 10.0),
    tau=0.06,
    seed=0,
):
    y_mc = np.asarray(y_mc, int)
    logo = LeaveOneGroupOut()
    X_modes = {fm: make_X_mode(X_raw, fm) for fm in feature_modes}

    oof = np.full_like(y_mc, -1, dtype=int)
    fold_scores = []
    chosen_rows = []

    for fold_id, (tr, te) in enumerate(logo.split(np.zeros_like(y_mc), groups=groups)):
        pred_te, cfg_rows, sc = multiclass_stack_onefold(
            tr, te, y_mc, groups,
            X_modes, feature_modes, inner_folds, multi_grid,
            topK=topK, meta_C_grid=meta_C_grid, tau=tau, seed=seed+fold_id
        )
        oof[te] = pred_te
        fold_scores.append(float(sc))
        for row in cfg_rows:
            row = dict(row)
            row["fold"] = int(fold_id)
            chosen_rows.append(row)

    if np.any(oof < 0):
        raise RuntimeError("Invalid OOF predictions (multi).")
    return oof, pd.DataFrame(chosen_rows), np.array(fold_scores, float)


In [ ]:
# ============================================================
# 7. 任务参数（k / delta / w）的处理：默认“固定参数”，避免任务定义泄漏
# ============================================================
# 说明（重要）：
# - V23 的 design-CV 会用全数据（含未来外层测试折）挑任务参数，严格口径下属于泄漏风险；
# - V24 默认改为“固定参数”（从PROFILE网格里取中位数），保证任务定义先验固定；
# - 若你确实要追求分数并接受“任务参数视作可学习设计变量”，可将 TASK_PARAM_MODE 改为 "nested"：
#   在每个外层LOCO折内，仅用训练集 + 内层GroupKFold选任务参数（不看测试折），仍然无泄漏。
#
# 推荐科研写法：论文主结果用 fixed；附录给出 nested 的上界对比（说明“可设计性”）。

TASK_PARAM_MODE = "nested"   # "fixed" 或 "nested"（每fold内选参更稳，但更慢）

def default_param_from_grid(grid):
    g = np.asarray(list(grid), float)
    return float(np.median(g))

# 固定参数（默认）
k_star = default_param_from_grid(PROFILE["notch_k"])
d_star = default_param_from_grid(PROFILE["double_delta"])
w_star = default_param_from_grid(PROFILE["piece_w"])
band_w_star = default_param_from_grid(PROFILE["band_w"])

print("Task param mode =", TASK_PARAM_MODE)
print(f"Fixed params: notch_k={k_star:.3g}, double_delta={d_star:.3g}, piece_w={w_star:.3g}, band_w={band_w_star:.3g}")

def select_param_nested_reg(tr_idx, param_grid, y_builder, feature_mode="fused"):
    """在外层训练折内（tr_idx）用内层GroupKFold选择回归任务参数。
    - 代理读出：Ridge(deg=2, alpha=1.0)
    - 评分：-MAE（越大越好）
    """
    inner = groupkfold_inner_splits(tr_idx, groups, PROFILE["inner_folds"])
    if inner is None:
        return float(param_grid[0])
    Xf = make_X_mode(X_raw, feature_mode)
    best_p, best_sc = None, -1e18
    for p in param_grid:
        y_all = y_builder(float(p))
        scs = []
        for itr, ite in inner:
            tr2 = tr_idx[itr]; te2 = tr_idx[ite]
            mdl = make_reg_pipeline("Ridge", degree=2, alpha=1.0)
            mdl.fit(Xf[tr2], y_all[tr2])
            pred = mdl.predict(Xf[te2])
            scs.append(-mean_absolute_error(y_all[te2], pred))
        sc = float(np.mean(scs)) if len(scs) else -1e18
        if sc > best_sc:
            best_sc, best_p = sc, float(p)
    return float(best_p)

def select_param_nested_bin(tr_idx, param_grid, y_builder, feature_mode="fused"):
    """在外层训练折内（tr_idx）用内层(优先Stratified)选择二分类任务参数。
    - 代理读出：LinearSVC(C=1.0)
    - 阈值：对 decision score 做训练内最优MCC阈值
    - 评分：phi-accuracy（越大越好）
    """
    # 先用每个候选p生成标签，再做分组+分层划分
    best_p, best_sc = None, -1e18
    Xf = make_X_mode(X_raw, feature_mode)
    for p in param_grid:
        yb = y_builder(float(p))
        inner = groupkfold_inner_splits(tr_idx, groups, PROFILE["inner_folds"], y=yb, stratify=True, seed=0)
        if inner is None:
            continue
        s_oof = np.full(len(tr_idx), np.nan, float)
        ok = True
        for itr, ite in inner:
            tr2 = tr_idx[itr]; te2 = tr_idx[ite]
            if len(np.unique(yb[tr2])) < 2:
                s_oof[ite] = float(np.mean(yb[tr2]))
                continue
            clf = make_binclf_pipeline("LinearSVC", degree=1, C=1.0)
            clf.fit(Xf[tr2], yb[tr2])
            s_oof[ite] = predict_continuous_score(clf, Xf[te2])
        if (not np.all(np.isfinite(s_oof))) or len(np.unique(yb[tr_idx])) < 2:
            continue
        thr, mcc = best_threshold_by_mcc(yb[tr_idx], s_oof)
        phi = phi_accuracy_from_mcc(mcc)
        if phi > best_sc:
            best_sc, best_p = float(phi), float(p)
    if best_p is None:
        return float(param_grid[0])
    return float(best_p)

In [ ]:

# ============================================================
# 8.1 ????????? + ??? + ???????
# ============================================================
_XMODE_CACHE = {}
_OUTER_SPLITS_CACHE = {}
_GROUPK_INNER_CACHE = {}

def clear_cached_features(*modes):
    """????????????????"""
    if not modes:
        _XMODE_CACHE.clear()
        return
    for m in modes:
        _XMODE_CACHE.pop(str(m), None)

def get_cached_X_modes(feature_modes, force_refresh=False):
    """??????????????????????? make_X_mode?"""
    out = {}
    for fm in feature_modes:
        fm = str(fm)
        need = force_refresh or (fm not in _XMODE_CACHE)
        # G_proxy ??????????????????
        if fm == "G_proxy":
            need = True
        if need:
            _XMODE_CACHE[fm] = np.asarray(make_X_mode(X_raw, fm), float)
        out[fm] = _XMODE_CACHE[fm]
    return out

def get_cached_outer_splits(y_like, groups):
    """????LOCO??????????? logo.split?"""
    g = np.asarray(groups)
    key = (len(np.asarray(y_like)), tuple(g.tolist()))
    if key not in _OUTER_SPLITS_CACHE:
        logo = LeaveOneGroupOut()
        _OUTER_SPLITS_CACHE[key] = list(logo.split(np.zeros(key[0]), groups=g))
    return _OUTER_SPLITS_CACHE[key]

def groupkfold_inner_splits(train_idx, groups, n_splits, y=None, stratify=False, seed=0):
    """????????????
    - ??????????? train_idx + groups ?????
    - ???????????????????????
    """
    train_idx = np.asarray(train_idx, int)
    g_all = np.asarray(groups)
    gtr = g_all[train_idx]
    n_groups = len(np.unique(gtr))
    ns = min(int(n_splits), int(n_groups))
    if ns < 2:
        return None

    if stratify and (y is not None):
        ytr = np.asarray(y)[train_idx]
        if len(np.unique(ytr)) >= 2:
            try:
                sgkf = StratifiedGroupKFold(n_splits=ns, shuffle=True, random_state=int(seed))
                return list(sgkf.split(np.zeros(len(train_idx)), ytr, groups=gtr))
            except Exception:
                pass
        gkf = GroupKFold(n_splits=ns)
        return list(gkf.split(np.zeros(len(train_idx)), groups=gtr))

    key = (ns, tuple(train_idx.tolist()), tuple(gtr.tolist()))
    if key in _GROUPK_INNER_CACHE:
        return _GROUPK_INNER_CACHE[key]

    gkf = GroupKFold(n_splits=ns)
    splits = list(gkf.split(np.zeros(len(train_idx)), groups=gtr))
    _GROUPK_INNER_CACHE[key] = splits
    return splits

def predict_multiclass_proba(clf, X):
    """?????????????(n_samples ? n_classes)????softmax???"""
    if hasattr(clf, "predict_proba"):
        P = clf.predict_proba(X)
        return np.asarray(P, float)
    if hasattr(clf, "decision_function"):
        S = np.asarray(clf.decision_function(X), float)
        if S.ndim == 1:
            P1 = 1.0 / (1.0 + np.exp(-S))
            return np.c_[1.0 - P1, P1]
        S = S - np.max(S, axis=1, keepdims=True)
        E = np.exp(S)
        return E / (np.sum(E, axis=1, keepdims=True) + 1e-12)
    y = clf.predict(X).astype(int)
    K = int(np.max(y)) + 1
    P = np.zeros((len(y), K), float)
    P[np.arange(len(y)), y] = 1.0
    return P

def select_param_nested_reg(tr_idx, param_grid, y_builder, feature_mode="fused"):
    """????????tr_idx????GroupKFold?????????"""
    inner = groupkfold_inner_splits(tr_idx, groups, PROFILE["inner_folds"])
    if inner is None:
        return float(param_grid[0])
    Xf = get_cached_X_modes([feature_mode])[feature_mode]
    best_p, best_sc = None, -1e18
    for p in param_grid:
        y_all = y_builder(float(p))
        scs = []
        for itr, ite in inner:
            tr2 = tr_idx[itr]
            te2 = tr_idx[ite]
            mdl = make_reg_pipeline("Ridge", degree=2, alpha=1.0)
            mdl.fit(Xf[tr2], y_all[tr2])
            pred = mdl.predict(Xf[te2])
            scs.append(-mean_absolute_error(y_all[te2], pred))
        sc = float(np.mean(scs)) if len(scs) else -1e18
        if sc > best_sc:
            best_sc, best_p = sc, float(p)
    return float(best_p)

def select_param_nested_bin(tr_idx, param_grid, y_builder, feature_mode="fused"):
    """????????tr_idx????(??Stratified)??????????"""
    best_p, best_sc = None, -1e18
    Xf = get_cached_X_modes([feature_mode])[feature_mode]
    for p in param_grid:
        yb = y_builder(float(p))
        inner = groupkfold_inner_splits(tr_idx, groups, PROFILE["inner_folds"], y=yb, stratify=True, seed=0)
        if inner is None:
            continue
        s_oof = np.full(len(tr_idx), np.nan, float)
        for itr, ite in inner:
            tr2 = tr_idx[itr]
            te2 = tr_idx[ite]
            if len(np.unique(yb[tr2])) < 2:
                s_oof[ite] = float(np.mean(yb[tr2]))
                continue
            clf = make_binclf_pipeline("LinearSVC", degree=1, C=1.0)
            clf.fit(Xf[tr2], yb[tr2])
            s_oof[ite] = predict_continuous_score(clf, Xf[te2])
        if (not np.all(np.isfinite(s_oof))) or len(np.unique(yb[tr_idx])) < 2:
            continue
        thr, mcc = best_threshold_by_mcc(yb[tr_idx], s_oof)
        phi = phi_accuracy_from_mcc(mcc)
        if phi > best_sc:
            best_sc, best_p = float(phi), float(p)
    if best_p is None:
        return float(param_grid[0])
    return float(best_p)

def nested_loco_regression_stack(
    y=None, y_reg=None, groups=None,
    feature_modes=None, inner_folds=5,
    reg_grid=None, topK=3,
    meta_alpha_grid=(0.1, 1.0, 10.0),
    mix_grid=(0.0, 0.25, 0.5, 0.75, 1.0),
    tau=0.06,
    select_metric="mae",
    calibrate_isotonic=False,
    X_modes=None,
    outer_splits=None,
):
    if y is None:
        y = y_reg
    y = np.asarray(y, float)
    if outer_splits is None:
        outer_splits = get_cached_outer_splits(y, groups)
    if X_modes is None:
        X_modes = get_cached_X_modes(feature_modes)

    oof = np.full_like(y, np.nan, dtype=float)
    fold_scores = []
    chosen_rows = []

    for fold_id, (tr, te) in enumerate(outer_splits):
        pred_te, cfg_rows, sc = regression_stack_onefold(
            tr, te, y, groups,
            X_modes, feature_modes, inner_folds, reg_grid,
            topK=topK,
            meta_alpha_grid=meta_alpha_grid,
            mix_grid=mix_grid,
            tau=tau,
            select_metric=select_metric,
            calibrate_isotonic=calibrate_isotonic,
        )
        oof[te] = pred_te
        fold_scores.append(float(sc))
        for row in cfg_rows:
            row = dict(row)
            row["fold"] = int(fold_id)
            chosen_rows.append(row)

    assert np.all(np.isfinite(oof))
    return oof, pd.DataFrame(chosen_rows), np.array(fold_scores, float)

def nested_loco_binary_stack(
    y_bin, groups, feature_modes, inner_folds, cls_grid, topK=3,
    meta_C_grid=(0.3, 1.0, 3.0, 10.0),
    tau=0.06,
    seed=0,
    X_modes=None,
    outer_splits=None,
):
    y_bin = np.asarray(y_bin, int)
    if outer_splits is None:
        outer_splits = get_cached_outer_splits(y_bin, groups)
    if X_modes is None:
        X_modes = get_cached_X_modes(feature_modes)

    oof = np.full_like(y_bin, -1, dtype=int)
    fold_rows = []
    chosen_rows = []

    for fold_id, (tr, te) in enumerate(outer_splits):
        pred_te, cfg_rows, fold_m = binary_stack_onefold(
            tr, te, y_bin, groups,
            X_modes, feature_modes, inner_folds, cls_grid,
            topK=topK, meta_C_grid=meta_C_grid, tau=tau, seed=seed+fold_id
        )
        oof[te] = pred_te
        group_val = groups[te][0] if hasattr(groups, "__len__") else None
        fold_rows.append(dict(fold=int(fold_id), group=group_val, **fold_m))
        for row in cfg_rows:
            row = dict(row)
            row["fold"] = int(fold_id)
            chosen_rows.append(row)

    if np.any(oof < 0):
        raise RuntimeError("Invalid OOF predictions (binary).")
    return oof, pd.DataFrame(chosen_rows), pd.DataFrame(fold_rows)

def nested_loco_multiclass(
    y_mc, groups, feature_modes, inner_folds, multi_grid, topK=3,
    meta_C_grid=(0.3, 1.0, 3.0, 10.0),
    tau=0.06,
    seed=0,
    X_modes=None,
    outer_splits=None,
):
    y_mc = np.asarray(y_mc, int)
    if outer_splits is None:
        outer_splits = get_cached_outer_splits(y_mc, groups)
    if X_modes is None:
        X_modes = get_cached_X_modes(feature_modes)

    oof = np.full_like(y_mc, -1, dtype=int)
    fold_scores = []
    chosen_rows = []

    for fold_id, (tr, te) in enumerate(outer_splits):
        pred_te, cfg_rows, sc = multiclass_stack_onefold(
            tr, te, y_mc, groups,
            X_modes, feature_modes, inner_folds, multi_grid,
            topK=topK, meta_C_grid=meta_C_grid, tau=tau, seed=seed+fold_id
        )
        oof[te] = pred_te
        fold_scores.append(float(sc))
        for row in cfg_rows:
            row = dict(row)
            row["fold"] = int(fold_id)
            chosen_rows.append(row)

    if np.any(oof < 0):
        raise RuntimeError("Invalid OOF predictions (multi).")
    return oof, pd.DataFrame(chosen_rows), np.array(fold_scores, float)

# ?????make_multiclf_pipeline????????????????
_SUPPORTED_MULTICLS_MODELS = {"LogReg", "LinearSVC", "LDA"}

def _multicls_model_name(item):
    if isinstance(item, dict):
        return item.get("model", item.get("name"))
    if isinstance(item, (tuple, list)) and len(item) >= 1:
        return item[0]
    return None

if "MULTICLS_GRID" in globals():
    _n_before = len(MULTICLS_GRID)
    MULTICLS_GRID = [it for it in MULTICLS_GRID if _multicls_model_name(it) in _SUPPORTED_MULTICLS_MODELS]
    _n_after = len(MULTICLS_GRID)
    print(f"[opt] MULTICLS_GRID filtered: {_n_before} -> {_n_after}")


In [ ]:

# ============================================================
# 8.5  G_proxy 构建（先用 LOCO 回归预测归一化浓度，再作为“代理特征”加入后续任务）
# 目的：对“由 G 定义的任务”（Stripes/SineSign/Bandpass/Quant/Ordinal 等）提升可分性
# 注意：这是一个两阶段策略（X -> G_hat -> task），可显著提升分类/多分类准确率
# ============================================================

# 开关：是否启用 G_proxy
PROFILE["prefer_G_proxy"] = True

if PROFILE.get("prefer_G_proxy", False):
    Gmin, Gmax = float(np.min(G)), float(np.max(G))
    G_norm = (G - Gmin) / (Gmax - Gmin + 1e-12)

    # 训练：用与主任务一致的“嵌套LOCO + Top-K stacking”回归来预测 G_norm
    g_oof, g_cfg, g_fold = nested_loco_regression_stack(
        y=G_norm, groups=groups,
        feature_modes=SEARCH_FEATURE_MODES,
        inner_folds=PROFILE["inner_folds"],
        reg_grid=REG_GRID,
        topK=PROFILE["topK_ensemble"],
        meta_alpha_grid=PROFILE["meta_ridge_alpha"],
        mix_grid=PROFILE.get("meta_mix_grid", (0.0, 0.25, 0.5, 0.75, 1.0)),
        tau=PROFILE["ensemble_tau"],
        select_metric="r2",
        calibrate_isotonic=True,
    )

    g_oof = np.clip(np.asarray(g_oof, float), 0.0, 1.0)

    # 代理特征：低维但信息密度高（可按需扩展）
    X_GPROXY = np.column_stack([
        g_oof,
        g_oof**2,
        np.sin(2*np.pi*g_oof),
        np.cos(2*np.pi*g_oof),
    ])

    # 把该模式加入搜索列表（后续任务的 feature_modes 会自动包含它）
    if "G_proxy" not in SEARCH_FEATURE_MODES:
        SEARCH_FEATURE_MODES = SEARCH_FEATURE_MODES + ["G_proxy"]

    # 评估：G_proxy 的 LOCO 预测误差（越小越好）
    G_hat = g_oof * (Gmax - Gmin) + Gmin
    g_mae = float(np.mean(np.abs(G_hat - G)))
    g_r2 = float(r2_score(G, G_hat))
    print(f"[G_proxy] LOCO: R2={g_r2:.4f}, MAE={g_mae:.4f} (G unit)")
else:
    print("[G_proxy] 未启用。")


In [ ]:
# ============================================================
# 8. 回归任务套件（严格LOCO评估 + Top-K stacking；结果/评估上下排列）
# ============================================================
logo = LeaveOneGroupOut()
outer_splits = list(logo.split(np.zeros(n), groups=groups))

def run_regression_curve(task_name, family, G0_list):
    """回归任务：支持两种模式
    - direct：对每个任务 y(G;G0,参数) 直接做回归（原逻辑）
    - proxy_analytic：先在外层LOCO得到浓度代理 G_hat（在前一节生成），再用解析任务函数 y=f(G_hat) 得到预测
      该方式通常更稳健、论文友好，且可显著降低漂移影响（优先推荐）。
    """
    rows = []
    cfg_rows = []
    cache = {}  # (G0) -> (y_true, y_pred)

    reg_mode = PROFILE.get("reg_mode", "direct")
    # DoubleTuning 对 G 误差非常敏感：默认使用 reg_mode_doubletuning（推荐 proxy_analytic）
    if family == "DoubleTuning" and PROFILE.get("force_direct_doubletuning", True):
        reg_mode = PROFILE.get("reg_mode_doubletuning", "proxy_analytic")


    # direct 模式需要预计算 X_modes
    X_modes = None
    if reg_mode == "direct":
        X_modes = get_cached_X_modes(SEARCH_FEATURE_MODES_REG)

    # proxy_analytic 需要 G_hat（前一节已在外层LOCO下得到 OOF 预测）
    if reg_mode == "proxy_analytic":
        if "G_hat" not in globals():
            raise RuntimeError("未找到 G_hat：请先运行 G_proxy 段落生成浓度代理预测。")
        Ghat = np.asarray(G_hat, dtype=float)

    for G0 in G0_list:
        G0 = float(G0)

        # =======================
        # 固定任务参数（更稳妥）
        # =======================
        if TASK_PARAM_MODE == "fixed":
            if family == "Notch":
                task_param = float(k_star)
                ytrue = notch_target(G0, k=task_param)
                if reg_mode == "proxy_analytic":
                    yhat = notch_on_G(Ghat, G0, k=task_param)
                else:
                    yhat, cfg, fold_sc = nested_loco_regression_stack(
                        y=ytrue, groups=groups,
                        feature_modes=SEARCH_FEATURE_MODES_REG,
                        inner_folds=PROFILE["inner_folds"],
                        reg_grid=REG_GRID,
                        topK=PROFILE["topK_ensemble"],
                        meta_alpha_grid=PROFILE["meta_ridge_alpha"],
                        mix_grid=PROFILE.get("meta_mix_grid", (0.0,0.25,0.5,0.75,1.0)),
                        tau=PROFILE["ensemble_tau"],
                        select_metric="r2",
                        X_modes=X_modes,
                        outer_splits=outer_splits,
                    )
            elif family == "DoubleTuning":
                task_param = float(d_star)
                ytrue = double_tuning_target(G0, delta_mM=task_param, k=6.0)
                if reg_mode == "proxy_analytic":
                    yhat = double_tuning_on_G(Ghat, G0, delta_mM=task_param, k=6.0)
                else:
                    yhat, cfg, fold_sc = nested_loco_regression_stack(
                        y=ytrue, groups=groups,
                        feature_modes=SEARCH_FEATURE_MODES_REG,
                        inner_folds=PROFILE["inner_folds"],
                        reg_grid=REG_GRID,
                        topK=PROFILE["topK_ensemble"],
                        meta_alpha_grid=PROFILE["meta_ridge_alpha"],
                        mix_grid=PROFILE.get("meta_mix_grid", (0.0,0.25,0.5,0.75,1.0)),
                        tau=PROFILE["ensemble_tau"],
                        select_metric="r2",
                        X_modes=X_modes,
                        outer_splits=outer_splits,
                    )
            else:
                task_param = float(w_star)
                ytrue = piecewise_saturation_target(G0, w_mM=task_param)
                if reg_mode == "proxy_analytic":
                    yhat = piecewise_sat_on_G(Ghat, G0, w_mM=task_param)
                else:
                    yhat, cfg, fold_sc = nested_loco_regression_stack(
                        y=ytrue, groups=groups,
                        feature_modes=SEARCH_FEATURE_MODES_REG,
                        inner_folds=PROFILE["inner_folds"],
                        reg_grid=REG_GRID,
                        topK=PROFILE["topK_ensemble"],
                        meta_alpha_grid=PROFILE["meta_ridge_alpha"],
                        mix_grid=PROFILE.get("meta_mix_grid", (0.0,0.25,0.5,0.75,1.0)),
                        tau=PROFILE["ensemble_tau"],
                        select_metric="r2",
                        X_modes=X_modes,
                        outer_splits=outer_splits,
                    )

            # fold_sc：用于“稳健性”统计（外层LOCO）
            if reg_mode == "proxy_analytic":
                fold_sc = []
                for tr, te in outer_splits:
                    fold_sc.append(float(mean_absolute_error(ytrue[te], yhat[te])))
                cfg = dict(
                    feature_mode="G_proxy",
                    model="ProxyAnalytic",
                    degree=-1,
                    params="{}",
                    inner_score=np.nan,
                    meta_alpha=np.nan,
                    mix=np.nan,
                )

            # 记录cfg

            if isinstance(cfg, pd.DataFrame):
                if len(cfg):
                    cfg_local = cfg.copy()
                    cfg_local["task"] = task_name
                    cfg_local["G0"] = G0
                    cfg_local["task_param_mode"] = TASK_PARAM_MODE
                    cfg_rows.extend(cfg_local.to_dict("records"))
            elif isinstance(cfg, list):
                for row in cfg:
                    row = dict(row)
                    row["task"] = task_name
                    row["G0"] = G0
                    row["task_param_mode"] = TASK_PARAM_MODE
                    cfg_rows.append(row)
            else:
                cfg = dict(cfg)
                cfg["task"] = task_name
                cfg["G0"] = G0
                cfg["task_param_mode"] = TASK_PARAM_MODE
                cfg_rows.append(cfg)

            met = regression_metrics(ytrue, yhat)
            rows.append(dict(
                task=task_name, G0=G0,
                r2=float(met["r2"]), mae=float(met["mae"]),
                fold_mean=float(np.mean(fold_sc)), fold_std=float(np.std(fold_sc)),
                task_param=float(task_param), task_param_mode=TASK_PARAM_MODE
            ))
            cache[G0] = (ytrue.copy(), yhat.copy())
            continue

        # ==========================================
        # nested 模式：每个外层fold用训练集内选择参数
        # ==========================================
        ytrue = np.full(n, np.nan, float)
        yhat = np.full(n, np.nan, float)
        fold_sc = []
        fold_p = []

        for fold_id, (tr, te) in enumerate(outer_splits):
            if family == "Notch":
                p_grid = PROFILE["notch_k"]
                # 目标函数：y=f(G)
                def f_onG(Gvals, p): return notch_on_G(Gvals, G0, k=float(p))
                def f_true(p): return notch_target(G0, k=float(p))
            elif family == "DoubleTuning":
                p_grid = PROFILE["double_delta"]
                def f_onG(Gvals, p): return double_tuning_on_G(Gvals, G0, delta_mM=float(p), k=6.0)
                def f_true(p): return double_tuning_target(G0, delta_mM=float(p), k=6.0)
            else:
                p_grid = PROFILE["piece_w"]
                def f_onG(Gvals, p): return piecewise_sat_on_G(Gvals, G0, w_mM=float(p))
                def f_true(p): return piecewise_saturation_target(G0, w_mM=float(p))

            # 选择p：只看训练集 tr
            if reg_mode == "proxy_analytic":
                best_p = None
                best_sc = None
                for p in p_grid:
                    yt = f_true(p)
                    yp = f_onG(Ghat, p)
                    sc = float(mean_absolute_error(yt[tr], yp[tr]))
                    if (best_sc is None) or (sc < best_sc):
                        best_sc = sc
                        best_p = float(p)
                p = float(best_p)
                y = f_true(p)
                pred_te = f_onG(Ghat, p)[te]
                yhat[te] = pred_te
                ytrue[te] = y[te]
                fold_sc.append(float(best_sc))
                fold_p.append(float(p))

                # cfg记录（简化）
                cfg_rows.append(dict(
                    rank=1, feature_mode="G_proxy", model="ProxyAnalytic", degree=-1,
                    params="{}", inner_score=np.nan, meta_alpha=np.nan, mix=np.nan,
                    fold=int(fold_id), task=task_name, G0=G0, task_param=float(p), task_param_mode=TASK_PARAM_MODE
                ))
            else:
                # direct：原逻辑（先选参数，再做回归stacking）
                if family == "Notch":
                    p = select_param_nested_reg(tr, p_grid, lambda kk: notch_target(G0, k=kk))
                    y = notch_target(G0, k=p)
                elif family == "DoubleTuning":
                    p = select_param_nested_reg(tr, p_grid, lambda dd: double_tuning_target(G0, delta_mM=dd, k=6.0))
                    y = double_tuning_target(G0, delta_mM=p, k=6.0)
                else:
                    p = select_param_nested_reg(tr, p_grid, lambda ww: piecewise_saturation_target(G0, w_mM=ww))
                    y = piecewise_saturation_target(G0, w_mM=p)

                pred_te, cfg_fold, sc = regression_stack_onefold(
                    tr, te, y, groups,
                    X_modes=X_modes, feature_modes=SEARCH_FEATURE_MODES_REG,
                    inner_folds=PROFILE["inner_folds"], reg_grid=REG_GRID,
                    topK=PROFILE["topK_ensemble"], meta_alpha_grid=PROFILE["meta_ridge_alpha"],
                    mix_grid=PROFILE.get("meta_mix_grid", (0.0,0.25,0.5,0.75,1.0)),
                    tau=PROFILE["ensemble_tau"], select_metric="r2"
                )
                yhat[te] = pred_te
                ytrue[te] = y[te]
                fold_sc.append(float(sc))
                fold_p.append(float(p))

                for row in cfg_fold:
                    row = dict(row)
                    row["fold"] = int(fold_id)
                    row["task"] = task_name
                    row["G0"] = G0
                    row["task_param"] = float(p)
                    row["task_param_mode"] = TASK_PARAM_MODE
                    cfg_rows.append(row)

        met = regression_metrics(ytrue, yhat)
        rows.append(dict(
            task=task_name, G0=G0,
            r2=float(met["r2"]), mae=float(met["mae"]),
            fold_mean=float(np.mean(fold_sc)), fold_std=float(np.std(fold_sc)),
            task_param_mean=float(np.mean(fold_p)), task_param_std=float(np.std(fold_p)),
            task_param_mode=TASK_PARAM_MODE
        ))
        cache[G0] = (ytrue.copy(), yhat.copy())

    cfg_df = pd.DataFrame(cfg_rows) if len(cfg_rows) else pd.DataFrame()
    return pd.DataFrame(rows).sort_values("G0"), cfg_df, cache


reg_families = [
    ("Notch", "Notch"),
    ("DoubleTuning", "DoubleTuning"),
    ("PiecewiseSat", "PiecewiseSat"),
]

reg_df_list = []
reg_cfg_list = []
reg_cache_all = {}

for task_name, fam in reg_families:
    df_one, cfg_one, cache_one = run_regression_curve(task_name, fam, G0_curve)
    reg_df_list.append(df_one.assign(task=task_name))
    reg_cfg_list.append(cfg_one)
    reg_cache_all[task_name] = cache_one

reg_df = pd.concat(reg_df_list, ignore_index=True)
reg_cfg_df = pd.concat(reg_cfg_list, ignore_index=True) if len(reg_cfg_list) else pd.DataFrame()

reg_df.to_csv(OUTDIR / "regression_curves_V41.csv", index=False)
reg_cfg_df.to_csv(OUTDIR / "regression_chosen_configs_V41.csv", index=False)

# 绘图：每个family一张（上：demo点预测；下：R2曲线）
def plot_reg_family(task_name):
    dd = reg_df[reg_df["task"] == task_name].sort_values("G0").reset_index(drop=True)
    G0_demo = float(dd.iloc[int(np.argmin(np.abs(dd["G0"].to_numpy(float) - float(demo_G0))))]["G0"])
    ytrue, yhat = reg_cache_all[task_name][G0_demo]

    fig = plt.figure(figsize=(7.2, 6.8), dpi=FIG_DPI)
    gs = fig.add_gridspec(2, 1, hspace=0.35)

    ax = fig.add_subplot(gs[0, 0])
    beautify_ax(ax)
    # 说明：回归任务上面板用“目标=线、预测=点”的形式。
    # 目标曲线用于展示任务定义的连续趋势；预测只画散点，避免视觉上暗示“预测必然平滑”。
    # 做法：按浓度G排序后连线；重复浓度点会形成竖向短线，属于正常现象。
    idx = np.argsort(G)
    Gs = G[idx]
    yt = ytrue[idx]
    yp = yhat[idx]
    ax.plot(Gs, yt, linewidth=2.2, label="Target")
    ax.scatter(Gs, yp, s=32, label="Prediction (LOCO OOF)", alpha=0.95)
    ax.axvline(G0_demo, linestyle="--", linewidth=1.6)
    ax.set_title(f"{task_name} demo (G0={G0_demo:.3g})")
    ax.set_xlabel("Glucose concentration (G)")
    ax.set_ylabel("Output")
    ax.legend()

    ax = fig.add_subplot(gs[1, 0])
    beautify_ax(ax)
    ax.plot(dd["G0"], dd["r2"], marker="o", linewidth=1.8, label="R2")
    ax.set_ylim(-0.2, 1.02)
    ax.set_title(f"{task_name} performance vs G0 (outer LOCO)")
    ax.set_xlabel("G0")
    ax.set_ylabel("R2")
    ax.legend()

    fn = OUTDIR / f"V43_reg_{task_name}.png"
    fig.savefig(fn, bbox_inches="tight")
    plt.close(fig)
    display(Image(filename=str(fn)))

for t in ["Notch", "DoubleTuning", "PiecewiseSat"]:
    plot_reg_family(t)

display(reg_df.sort_values(["task","G0"]).head())

In [ ]:
# ============================================================
# 9. 二分类任务套件（严格LOCO评估 + Top-K stacking + 阈值训练内优化）
# ============================================================
logo = LeaveOneGroupOut()
outer_splits = list(logo.split(np.zeros(n), groups=groups))

# 预计算X_modes
X_modes_cls = get_cached_X_modes(SEARCH_FEATURE_MODES)

# -------------------------
# 9.1 固定任务（Stripes / SineSign）
# -------------------------
cls_tasks = [
    ("Stripes_m4", stripes_target(m=4)),
    ("SineSign_m2", sine_sign_target(m=2)),
]

cls_rows, cls_cfg_rows, cls_fold_rows = [], [], []
cls_oof_cache = {}
cls_fold_cache = {}

for task_name, yb in cls_tasks:
    oof, cfg, fold_df = nested_loco_binary_stack(
        y_bin=yb, groups=groups,
        X_modes=X_modes_cls,
        outer_splits=outer_splits,
        feature_modes=SEARCH_FEATURE_MODES,
        inner_folds=PROFILE["inner_folds"],
        cls_grid=BINCLS_GRID,
        topK=PROFILE["topK_ensemble"],
        meta_C_grid=PROFILE["meta_logreg_C"],
        tau=PROFILE["ensemble_tau"],
        seed=0,
    )
    met = binary_metrics(yb, oof)

    cls_rows.append(dict(
        task=task_name,
        phi_acc=met["phi_acc"],
        bal_acc=met["bal_acc"],
        fold_acc_mean=float(fold_df["acc"].mean()),
        fold_acc_std=float(fold_df["acc"].std(ddof=0)),
    ))

    cfg["task"] = task_name
    cls_cfg_rows.append(cfg)

    for _, r in fold_df.iterrows():
        cls_fold_rows.append(dict(task=task_name, fold=int(r["fold"]), group=r["group"], acc=float(r["acc"]), bal_acc=float(r["bal_acc"]), phi_acc=float(r["phi_acc"])))

    cls_oof_cache[task_name] = oof
    cls_fold_cache[task_name] = fold_df["acc"].to_numpy(float)

cls_df = pd.DataFrame(cls_rows)
pd.concat(cls_cfg_rows, ignore_index=True).to_csv(OUTDIR / "classification_chosen_configs_V43.csv", index=False)
cls_df.to_csv(OUTDIR / "classification_scores_V43.csv", index=False)
pd.DataFrame(cls_fold_rows).to_csv(OUTDIR / "classification_fold_scores_V43.csv", index=False)
display(cls_df)

# -------------------------
# 9.1.1 固定二分类任务：主图(均值+95%CI) + 折分布(箱线+抖动)
# -------------------------
# 说明：上方主图给出均值与bootstrap 95%CI；下方展示外层折分布。


tasks_fixed = [t for t, _ in cls_tasks]
fold_vals = [np.array(cls_fold_cache[t], float) for t in tasks_fixed]

# --- bootstrap 95%CI（对外层折均值）
rng = np.random.default_rng(0)
boot_n = 2000
ci_low = []
ci_high = []
means = []
for vals in fold_vals:
    vals = np.asarray(vals, float)
    means.append(float(np.mean(vals)))
    if len(vals) == 0:
        ci_low.append(np.nan); ci_high.append(np.nan); continue
    boots = rng.choice(vals, size=(boot_n, len(vals)), replace=True).mean(axis=1)
    ci_low.append(float(np.percentile(boots, 2.5)))
    ci_high.append(float(np.percentile(boots, 97.5)))

fig = plt.figure(figsize=(6.8, 5.6), dpi=FIG_DPI)

# 主图：均值 + 95%CI
ax = fig.add_subplot(2, 1, 1)
beautify_ax(ax)
xs = np.arange(len(tasks_fixed)) + 1
ax.errorbar(xs, means, yerr=[np.array(means) - np.array(ci_low), np.array(ci_high) - np.array(means)],
            fmt="o", capsize=4, label="Mean ±95% CI")
ax.set_xticks(xs)
ax.set_xticklabels(tasks_fixed)
ax.set_ylim(0.0, 1.02)
ax.set_ylabel("Accuracy")
ax.set_title("Fixed binary tasks (mean ±95% CI)")
ax.legend()

# 稳健性图：箱线 + 抖动散点
ax = fig.add_subplot(2, 1, 2)
beautify_ax(ax)
ax.boxplot(fold_vals, labels=tasks_fixed, showfliers=False)
for i, vals in enumerate(fold_vals, start=1):
    x = rng.normal(i, 0.06, size=len(vals))
    ax.scatter(x, vals, s=16, alpha=0.6, color="tab:blue")
ax.set_xlabel("Task")
ax.set_ylabel("Accuracy")
ax.set_ylim(0.0, 1.02)
ax.set_title("Outer LOCO fold distribution")

fn = OUTDIR / "V43_fixed_binary_ci_boxplot.png"
fig.savefig(fn, bbox_inches="tight")
plt.close(fig)
display(Image(filename=str(fn)))


# -------------------------
# 9.2 Band-pass：随 G0 扫描（宽度 fixed 或 nested）
# -------------------------
band_rows = []
band_cache = {}
band_acc_cache = {}  # key=G0, value=list of fold accuracy (length = #outer folds)

for G0 in G0_curve:
    G0 = float(G0)

    if TASK_PARAM_MODE == "fixed":
        yb = bandpass_target(G0, w_mM=float(band_w_star))
        oof, cfg, fold_df = nested_loco_binary_stack(
            y_bin=yb, groups=groups,
            X_modes=X_modes_cls,
            outer_splits=outer_splits,
            feature_modes=SEARCH_FEATURE_MODES,
            inner_folds=PROFILE["inner_folds"],
            cls_grid=BINCLS_GRID,
            topK=PROFILE["topK_ensemble"],
            meta_C_grid=PROFILE["meta_logreg_C"],
            tau=PROFILE["ensemble_tau"],
            seed=11,
        )
        met = binary_metrics(yb, oof)
        band_rows.append(dict(task="Bandpass", G0=G0, phi_acc=met["phi_acc"], bal_acc=met["bal_acc"],
                              fold_acc_mean=float(fold_df["acc"].mean()), fold_acc_std=float(fold_df["acc"].std(ddof=0)),
                              w=float(band_w_star), task_param_mode=TASK_PARAM_MODE))
        band_cache[G0] = (yb.copy(), oof.copy(), np.full(len(fold_df), float(band_w_star)))
        band_acc_cache[G0] = list(map(float, fold_df["phi_acc"].values))
        continue

    # nested：每fold用训练集内选w
    ytrue = np.full(n, -1, int)
    ypred = np.full(n, -1, int)
    fold_phi = []  # per-fold accuracy
    fold_w = []
    cfg_list = []

    for fold_id, (tr, te) in enumerate(outer_splits):
        w = select_param_nested_bin(tr, PROFILE["band_w"], lambda ww: bandpass_target(G0, w_mM=ww))
        yb = bandpass_target(G0, w_mM=float(w))

        pred_te, cfg_fold, fold_m = binary_stack_onefold(
            tr, te, yb, groups,
            X_modes=X_modes_cls, feature_modes=SEARCH_FEATURE_MODES,
            inner_folds=PROFILE["inner_folds"], cls_grid=BINCLS_GRID,
            topK=PROFILE["topK_ensemble"], meta_C_grid=PROFILE["meta_logreg_C"],
            tau=PROFILE["ensemble_tau"], seed=11+fold_id
        )
        ytrue[te] = yb[te]
        ypred[te] = pred_te
        fold_phi.append(float(fold_m['acc']))
        fold_w.append(float(w))
        for row in cfg_fold:
            row = dict(row)
            row["fold"] = int(fold_id); row["task"] = "Bandpass"; row["G0"] = G0
            row["w"] = float(w); row["task_param_mode"] = TASK_PARAM_MODE
            cfg_list.append(pd.DataFrame([row]))

    met = binary_metrics(ytrue[ytrue>=0], ypred[ypred>=0])
    band_rows.append(dict(task="Bandpass", G0=G0, phi_acc=met["phi_acc"], bal_acc=met["bal_acc"],
                          fold_acc_mean=float(np.mean(fold_phi)), fold_acc_std=float(np.std(fold_phi)),
                          w_mean=float(np.mean(fold_w)), w_std=float(np.std(fold_w)),
                          task_param_mode=TASK_PARAM_MODE))
    band_cache[G0] = (ytrue.copy(), ypred.copy(), np.array(fold_w, float))
    band_acc_cache[G0] = list(map(float, fold_phi))

band_df = pd.DataFrame(band_rows).sort_values("G0")
band_df.to_csv(OUTDIR / "bandpass_curve_V43.csv", index=False)

# 绘图：上（demo输出 vs G），下（phi vs G0）
G0_demo = float(G0_curve[int(len(G0_curve)//2)])
G0_demo = float(band_df.iloc[int(np.argmin(np.abs(band_df["G0"].to_numpy(float) - float(demo_G0))))]["G0"])
ytrue_demo, ypred_demo, w_fold_demo = band_cache[G0_demo]

fig = plt.figure(figsize=(7.2, 6.8), dpi=FIG_DPI)
gs = fig.add_gridspec(2, 1, hspace=0.35)

ax = fig.add_subplot(gs[0, 0])
# 轻微jitter，避免0/1重叠遮挡
rng = np.random.default_rng(0)
jitter = 0.03
ax.scatter(G, ytrue_demo + rng.normal(0, jitter, size=len(G)), s=28, label="Target", alpha=0.75)
ax.scatter(G, ypred_demo + rng.normal(0, jitter, size=len(G)), s=28, label="Prediction (LOCO OOF)", alpha=0.75)
ax.axvline(G0_demo, linestyle="--", linewidth=1.6)
ax.set_title(f"Band-pass demo (G0={G0_demo:.3g})")
ax.set_xlabel("Glucose concentration (G)")
ax.set_ylabel("Class (0/1)")
ax.legend()

ax = fig.add_subplot(gs[1, 0])
beautify_ax(ax)
G0_vals = band_df["G0"].to_numpy(float)
# 分位数带（更稳健）
M = np.vstack([np.array(band_acc_cache[float(g)], float) for g in G0_vals])
med_vals = np.nanmedian(M, axis=1)
q25 = np.nanpercentile(M, 25, axis=1)
q75 = np.nanpercentile(M, 75, axis=1)
ax.plot(G0_vals, med_vals, marker="o", linewidth=1.8, label="Median")
ax.fill_between(G0_vals, q25, q75, alpha=0.25, label="25–75% band")
ax.axhline(0.5, linestyle="--", color="gray", linewidth=1.2, label="Baseline 0.5")
ax.set_title("Band-pass accuracy vs G0 (outer LOCO)")
ax.set_xlabel("G0")
ax.set_ylabel("Accuracy")
ax.set_ylim(0.0, 1.02)
ax.legend()

fn = OUTDIR / "V43_bandpass.png"
fig.savefig(fn, bbox_inches="tight")
plt.close(fig)
display(Image(filename=str(fn)))

# 追加：折×G0 稳健性热图
fig = plt.figure(figsize=(7.2, 3.2), dpi=FIG_DPI)
ax = fig.add_subplot(1, 1, 1)
beautify_ax(ax)
M_fold = np.vstack([np.array(band_acc_cache[float(g)], float) for g in G0_vals]).T
im = ax.imshow(M_fold, aspect="auto", origin="lower",
               extent=[float(G0_vals.min()), float(G0_vals.max()), -0.5, M_fold.shape[0]-0.5],
               vmin=0.0, vmax=1.0, cmap="turbo")
ax.set_title("Band-pass fold × G0 accuracy map")
ax.set_xlabel("G0")
ax.set_ylabel("Outer LOCO fold")
cb = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
cb.set_label("Accuracy")

fn = OUTDIR / "V43_bandpass_foldmap.png"
fig.savefig(fn, bbox_inches="tight")
plt.close(fig)
display(Image(filename=str(fn)))

display(band_df.head())

In [ ]:

# ============================================================
# 10. 多级任务：Quantization(K=4) + Ordinal（严格LOCO + Top-K集成）
# ============================================================
K = 4
yq, edges = quantize_classes(K=K)
ys_ord = ordinal_targets_from_edges(edges)

# ???X_modes???Quantization + Ordinal???????
X_modes_multi = get_cached_X_modes(SEARCH_FEATURE_MODES)

# 10.1 Quantization（多分类）
oof_q, cfg_q, f1_fold = nested_loco_multiclass(
    y_mc=yq, groups=groups,
    X_modes=X_modes_multi,
    feature_modes=SEARCH_FEATURE_MODES,
    inner_folds=PROFILE["inner_folds"],
    multi_grid=MULTICLS_GRID,
    topK=PROFILE["topK_ensemble"], meta_C_grid=PROFILE["meta_logreg_C"], tau=PROFILE["ensemble_tau"]
)
met_q = multiclass_metrics(yq, oof_q)
cm_q = confusion_matrix(yq, oof_q)

# 10.2 Ordinal（序数）：K-1个二分类阈值输出再汇总为等级
oof_bins = []
for yb in ys_ord:
    oof_b, cfg_b, phi_fold = nested_loco_binary_stack(
        y_bin=yb, groups=groups,
        X_modes=X_modes_multi,
        feature_modes=SEARCH_FEATURE_MODES,
        inner_folds=PROFILE["inner_folds"],
        cls_grid=BINCLS_GRID,
        topK=PROFILE["topK_ensemble"], meta_C_grid=PROFILE["meta_logreg_C"], tau=PROFILE["ensemble_tau"]
    )
    oof_bins.append(oof_b)

oof_bins = np.vstack(oof_bins)
y_ord_pred = np.sum(oof_bins, axis=0).astype(int)
y_ord_true = yq.copy()

met_ord = dict(
    macro_f1=float(f1_score(y_ord_true, y_ord_pred, average="macro")),
    bal_acc=float(balanced_accuracy_score(y_ord_true, y_ord_pred)),
    mae_level=float(mean_absolute_error(y_ord_true, y_ord_pred))
)
cm_ord = confusion_matrix(y_ord_true, y_ord_pred)

# 保存
pd.DataFrame([met_q]).to_csv(OUTDIR / "quantization_metrics_V43.csv", index=False)
pd.DataFrame([met_ord]).to_csv(OUTDIR / "ordinal_metrics_V43.csv", index=False)
pd.DataFrame(cm_q).to_csv(OUTDIR / "quantization_confusion_V43.csv", index=False, header=False)
pd.DataFrame(cm_ord).to_csv(OUTDIR / "ordinal_confusion_V43.csv", index=False, header=False)

# 绘图：上下排列（confusion / metrics）
def plot_multilevel(cm, title, metrics_dict, fn):
    fig = plt.figure(figsize=(7.2, 6.8), dpi=FIG_DPI)
    gs = fig.add_gridspec(2, 1, hspace=0.35)

    ax = fig.add_subplot(gs[0, 0])
    beautify_ax(ax)
    ax.imshow(cm, aspect="auto", cmap="turbo")
    ax.set_title(title + " confusion")
    ax.set_xlabel("Pred")
    ax.set_ylabel("True")

    ax = fig.add_subplot(gs[1, 0])
    beautify_ax(ax)
    keys = list(metrics_dict.keys())
    vals = [metrics_dict[k] for k in keys]
    # 为了统一“越大越好”，对MAE做一个简单映射：score = 1 - MAE/(K-1)
    vals2 = []
    for k,v in zip(keys, vals):
        if "mae" in k:
            vals2.append(1.0 - min(1.0, float(v)/(K-1)))
        else:
            vals2.append(float(v))
    ax.bar(keys, vals2)
    ax.set_ylim(0.0, 1.02)
    ax.set_title(title + " metrics (LOCO)")
    ax.set_ylabel("Score (higher is better)")

    fig.savefig(fn, bbox_inches="tight")
    plt.close(fig)
    display(Image(filename=str(fn)))

plot_multilevel(cm_q, "Quantization (K=4)", met_q, OUTDIR / "V43_multilevel_quantization.png")
plot_multilevel(cm_ord, "Ordinal", met_ord, OUTDIR / "V43_multilevel_ordinal.png")

print("Quantization:", met_q)
print("Ordinal:", met_ord)


In [ ]:
# ============================================================
# 11. Pairwise ranking（与LOCO一致：外层留一浓度；测试对为“测试样本 vs 训练样本”）
# ============================================================
rng = np.random.default_rng(42)


def sample_pairs_cross(test_idx, train_idx, max_pairs=2000, rng=None):
    """构造跨集合pairs：每个test样本与若干train样本配对（只取“test -> train”方向）。
    关键修复点：
    - 之前pairs方向固定且与样本索引顺序相关，容易导致标签单一（AUC无法计算）。
    - 这里保证：当训练集中同时存在“比test更小/更大”的浓度时，优先各采一部分，
      避免抽样后恰好只剩单一类别。
    """
    rng = np.random.default_rng(0) if rng is None else rng
    test_idx = np.asarray(test_idx, int)
    train_idx = np.asarray(train_idx, int)

    pairs = []
    for i in test_idx:
        gi = G[i]
        lower = train_idx[G[train_idx] < gi]
        higher = train_idx[G[train_idx] > gi]

        # 目标：尽量同时抽到 lower 与 higher（这样 yte 同时包含 0/1）
        quota = max_pairs // max(1, len(test_idx))
        quota = max(20, int(quota))  # 每个test至少抽一些，避免过稀

        if len(lower) and len(higher):
            n1 = quota // 2
            n2 = quota - n1
            js1 = rng.choice(lower, size=min(n1, len(lower)), replace=False)
            js2 = rng.choice(higher, size=min(n2, len(higher)), replace=False)
            js = np.concatenate([js1, js2])
        else:
            # 极端浓度：只有一侧可比（此时 AUC 在该折可能不可定义，这是客观限制）
            pool = lower if len(lower) else higher
            if len(pool) == 0:
                continue
            js = rng.choice(pool, size=min(quota, len(pool)), replace=False)

        for j in js:
            pairs.append((i, int(j)))

    if len(pairs) == 0:
        return np.empty((0, 2), int)

    # 全局截断
    if len(pairs) > max_pairs:
        pairs = rng.choice(pairs, size=max_pairs, replace=False).tolist()

    return np.array(pairs, int)


def sample_pairs_within(idx, max_pairs=4000, rng=None):
    """构造集合内部pairs（训练用）。
    关键修复点：
    - 之前只取 (i,j) 且强依赖样本索引顺序；如果数据按浓度排序，标签会几乎全为0或全为1，
      导致 ytr 单类，从而所有外层折都被跳过（folds_used=0）。
    - 现在对每个无序对 {i,j} 同时加入 (i,j) 与 (j,i)，保证标签天然成对互补，AUC可计算。
    """
    rng = np.random.default_rng(0) if rng is None else rng
    idx = np.asarray(idx, int)
    pairs = []
    for a in range(len(idx)):
        for b in range(a + 1, len(idx)):
            i, j = int(idx[a]), int(idx[b])
            if G[i] == G[j]:
                continue
            pairs.append((i, j))
            pairs.append((j, i))

    if len(pairs) == 0:
        return np.empty((0, 2), int)

    if len(pairs) > max_pairs:
        pairs = rng.choice(pairs, size=max_pairs, replace=False).tolist()

    return np.array(pairs, int)

def build_pair_dataset(pairs, feature_mode="fused"):
    Xi = make_X_mode(X_raw[pairs[:, 0]], feature_mode)
    Xj = make_X_mode(X_raw[pairs[:, 1]], feature_mode)
    Xd = Xi - Xj
    y = (G[pairs[:, 0]] > G[pairs[:, 1]]).astype(int)
    return Xd, y

def make_rank_model(C=1.0):
    return Pipeline([
        ("scaler", StandardScaler()),
        ("lr", LogisticRegression(C=float(C), max_iter=30000, solver="liblinear", class_weight="balanced")),
    ])

rank_C_grid = [0.2, 1.0, 5.0, 20.0]
logo = LeaveOneGroupOut()

auc_folds, acc_folds = [], []

for fold_id, (tr, te) in enumerate(logo.split(np.zeros(n), groups=groups)):
    pairs_tr = sample_pairs_within(tr, max_pairs=int(PROFILE["rank_max_pairs_train"]), rng=rng)
    pairs_te = sample_pairs_cross(te, tr, max_pairs=int(PROFILE["rank_max_pairs_test"]), rng=rng)
    if len(pairs_tr) < int(PROFILE["rank_min_pairs"]) or len(pairs_te) < int(PROFILE["rank_min_pairs"]//2):
        continue

    Xtr, ytr = build_pair_dataset(pairs_tr, feature_mode="fused")
    Xte, yte = build_pair_dataset(pairs_te, feature_mode="fused")
    if len(np.unique(ytr)) < 2 or len(np.unique(yte)) < 2:
        continue

    # -------------------------------
    # 内层：为pairwise ranking选择超参数C（严格在外层训练集内完成）
    # 关键修复：GroupShuffleSplit 必须传入 groups
    # 这里为“每一对(pair)”构造 pair-level groups：使用两端样本所属浓度组的无序组合编码，
    # 使得同一种“浓度组合”的pairs不会被拆到不同子集中。
    # -------------------------------
    gg1 = groups[pairs_tr[:, 0]].astype(int)
    gg2 = groups[pairs_tr[:, 1]].astype(int)
    lo = np.minimum(gg1, gg2)
    hi = np.maximum(gg1, gg2)
    maxg = int(np.max(groups)) + 1
    pair_groups = (lo * maxg + hi).astype(int)

    # 预先生成内层划分，保证不同C使用同一组划分（可重复、可比较）
    if len(np.unique(pair_groups)) >= 2:
        inner = GroupShuffleSplit(n_splits=6, test_size=0.25, random_state=fold_id+1)
        inner_splits = list(inner.split(Xtr, ytr, groups=pair_groups))
    else:
        # 极少见：若训练集中只出现一种“浓度组合”，则分组划分无意义，退化为分层随机划分
        from sklearn.model_selection import StratifiedShuffleSplit
        inner = StratifiedShuffleSplit(n_splits=6, test_size=0.25, random_state=fold_id+1)
        inner_splits = list(inner.split(Xtr, ytr))

    best_C, best_auc = None, -1e18
    for C in rank_C_grid:
        aucs = []
        for tr2, va in inner_splits:
            if len(np.unique(ytr[tr2])) < 2 or len(np.unique(ytr[va])) < 2:
                continue
            clf = make_rank_model(C=C)
            clf.fit(Xtr[tr2], ytr[tr2])
            pva = clf.predict_proba(Xtr[va])[:, 1]
            aucs.append(roc_auc_score(ytr[va], pva))
        if len(aucs):
            sc = float(np.mean(aucs))
            if sc > best_auc:
                best_auc, best_C = sc, float(C)

    if best_C is None:
        continue

    clf = make_rank_model(C=best_C)
    clf.fit(Xtr, ytr)
    pte = clf.predict_proba(Xte)[:, 1]
    auc = float(roc_auc_score(yte, pte))
    pred = (pte >= 0.5).astype(int)
    acc = float(np.mean(pred == yte))

    auc_folds.append(auc)
    acc_folds.append(acc)

rank_metrics = dict(
    auc_mean=float(np.mean(auc_folds)) if len(auc_folds) else np.nan,
    auc_std=float(np.std(auc_folds)) if len(auc_folds) else np.nan,
    acc_mean=float(np.mean(acc_folds)) if len(acc_folds) else np.nan,
    acc_std=float(np.std(acc_folds)) if len(acc_folds) else np.nan,
    n_folds_used=int(len(auc_folds)),
)
pd.DataFrame([rank_metrics]).to_csv(OUTDIR / "pairwise_ranking_metrics_V43.csv", index=False)

# 绘图：上下排列（AUC分布 / summary）
fig = plt.figure(figsize=(7.2, 6.8), dpi=FIG_DPI)
gs = fig.add_gridspec(2, 1, hspace=0.35)

ax = fig.add_subplot(gs[0, 0])
if len(auc_folds):
    ax.hist(auc_folds, bins=10)
else:
    ax.text(0.5, 0.5, 'No valid folds (AUC undefined under current LOCO setting)', ha='center', va='center', transform=ax.transAxes)
ax.set_title("Pairwise ranking AUC (outer LOCO)")
ax.set_xlabel("AUC")
ax.set_ylabel("Count")

ax = fig.add_subplot(gs[1, 0])
ax.bar(["AUC mean", "ACC mean"], [rank_metrics["auc_mean"], rank_metrics["acc_mean"]])
ax.set_ylim(0.0, 1.02)
ax.set_title(f"Ranking summary (folds used={rank_metrics['n_folds_used']})")
ax.set_ylabel("Score")

fn = OUTDIR / "V43_ranking.png"
fig.savefig(fn, bbox_inches="tight")
plt.close(fig)
display(Image(filename=str(fn)))

print("Ranking metrics (V43):", rank_metrics)

In [ ]:
# ============================================================
# 12. 特征子集规模曲线（容量分析；修正x轴含义：k表示“选取的峰数量”）
# ============================================================
RUN_SUBSET = True
subset_rows = []
rng = np.random.default_rng(123)
logo = LeaveOneGroupOut()

# 代表任务：Notch@demo_G0 + Stripes（固定任务参数，避免与主体结果混淆）
y_reg = notch_target(float(demo_G0), k=float(k_star))
y_cls = stripes_target(m=4)

def loco_reg_fixed(Xsub, y):
    pred = np.zeros_like(y, dtype=float)
    for tr, te in logo.split(Xsub, y, groups=groups):
        model = make_reg_pipeline("Ridge", degree=2, alpha=1.0)
        model.fit(Xsub[tr], y[tr])
        pred[te] = model.predict(Xsub[te])
    return pred

def loco_cls_fixed(Xsub, yb):
    pred = np.zeros_like(yb, dtype=int)
    for tr, te in logo.split(Xsub, yb, groups=groups):
        if len(np.unique(yb[tr])) < 2:
            pred[te] = int(np.round(yb[tr].mean()))
            continue
        clf = make_binclf_pipeline("LinearSVC", degree=1, C=1.0)
        clf.fit(Xsub[tr], yb[tr])
        pred[te] = clf.predict(Xsub[te]).astype(int)
    return pred

if RUN_SUBSET:
    for k in PROFILE["subset_ks"]:
        for rep in range(PROFILE["subset_repeats"]):
            cols = rng.choice(p, size=int(k), replace=False)

            # 关键修正：
            # - 这里研究“选取k个峰（原始峰特征）”对解码性能的影响；
            # - 因此Xsub直接使用 raw 子集，避免 fused 使特征维数变为 3k+1 导致x轴误解。
            Xsub = X_raw[:, cols].astype(float)

            pr = loco_reg_fixed(Xsub, y_reg)
            mr = regression_metrics(y_reg, pr)

            pc = loco_cls_fixed(Xsub, y_cls)
            mc = binary_metrics(y_cls, pc)

            subset_rows.append(dict(
                k=int(k), rep=int(rep),
                reg_r2=mr["r2"], reg_mae=mr["mae"],
                cls_phi_acc=mc["phi_acc"], cls_bal_acc=mc["bal_acc"]
            ))

subset_df = pd.DataFrame(subset_rows)
subset_df.to_csv(OUTDIR / "feature_subset_capacity_V43.csv", index=False)

# 绘图：上下排列（均值曲线 / 误差条）
if len(subset_df):
    agg = subset_df.groupby("k").agg(
        r2_mean=("reg_r2", "mean"), r2_std=("reg_r2", "std"),
        phi_mean=("cls_phi_acc", "mean"), phi_std=("cls_phi_acc", "std")
    ).reset_index()

    fig = plt.figure(figsize=(7.2, 6.8), dpi=FIG_DPI)
    gs = fig.add_gridspec(2, 1, hspace=0.35)

    ax = fig.add_subplot(gs[0, 0])
    beautify_ax(ax)
    ax.plot(agg["k"], agg["r2_mean"], marker="o", linewidth=1.8, label="Regression R2")
    ax.plot(agg["k"], agg["phi_mean"], marker="o", linewidth=1.8, label="Classification Phi-acc")
    ax.set_ylim(-0.2, 1.02)
    ax.set_title("Subset capacity (mean)")
    ax.set_xlabel("#selected peaks (k)")
    ax.set_ylabel("Score")
    ax.legend()

    ax = fig.add_subplot(gs[1, 0])
    beautify_ax(ax)
    ax.errorbar(agg["k"], agg["r2_mean"], yerr=agg["r2_std"], marker="o", capsize=3, label="R2 ± std")
    ax.errorbar(agg["k"], agg["phi_mean"], yerr=agg["phi_std"], marker="o", capsize=3, label="Phi ± std")
    ax.set_ylim(-0.2, 1.02)
    ax.set_title("Subset capacity (mean ± std)")
    ax.set_xlabel("#selected peaks (k)")
    ax.set_ylabel("Score")
    ax.legend()

    fn = OUTDIR / "V43_feature_capacity.png"
    fig.savefig(fn, bbox_inches="tight")
    plt.close(fig)
    display(Image(filename=str(fn)))

print("Subset rows:", len(subset_df))

In [ ]:

# ============================================================
# 13. 通用顶刊式结果展示（2×2面板 + 全任务记分板）
#    目标：
#    - 每个任务族：2×2 面板（任务定义 / 代表性demo / 性能谱 / 稳健性）
#    - 全任务：统一“记分板”（表格+图），便于写论文与横向对比
#    注意：
#    - 图例与坐标轴全部英文（避免字体问题）
#    - 代码注释尽量中文，便于你后续维护
# ============================================================

# -------------------------
# 13.0 外层LOCO折（后续稳健性统计需要）
# -------------------------
logo = LeaveOneGroupOut()
outer_splits = list(logo.split(np.zeros(n), groups=groups))
n_outer = len(outer_splits)

# -------------------------
# 13.1 回归族：Notch / DoubleTuning / PiecewiseSat
# -------------------------
def reg_target_on_grid(family, G0, G_grid):
    """在给定的浓度网格 G_grid 上计算任务目标曲线。
    说明：
    - 原 notebook 中的 notch_target / double_tuning_target / piecewise_saturation_target
      是在“样本点”上计算（依赖全局 x 数组）。
    - 为了画“任务定义曲线”，这里改为在指定G_grid上计算，避免维度不匹配。
    """
    G0 = float(G0)
    G_grid = np.asarray(G_grid, float)
    xg = (G_grid - G_min) / (G_max - G_min + 1e-12)
    x0 = (G0 - G_min) / (G_max - G_min + 1e-12)

    if family == "Notch":
        return 1.0 - tuning_target_from_x(xg, x0, float(k_star))
    if family == "DoubleTuning":
        d = float(d_star) / (G_max - G_min + 1e-12)
        y = tuning_target_from_x(xg, x0 - d, 6.0) + tuning_target_from_x(xg, x0 + d, 6.0)
        mx = np.max(y)
        return (y / mx) if mx > 0 else y
    # PiecewiseSat
    w = float(w_star) / (G_max - G_min + 1e-12)
    y = (xg - x0) / (w + 1e-12) + 0.5
    return np.clip(y, 0.0, 1.0)

def reg_fold_mae(y_true, y_pred):
    """按外层折计算MAE列表，用于箱线图/误差带（比R2更稳健）。"""
    maes = []
    for tr, te in outer_splits:
        maes.append(float(mean_absolute_error(y_true[te], y_pred[te])))
    return np.array(maes, float)

def plot_reg_panel(family, G0_demo=25.0):
    """为某个回归族绘制 2×2 面板。"""
    # 1) 性能谱数据：从 reg_cache_all 读取每个 G0 的 (y_true, y_pred)
    g0s = sorted(reg_cache_all[family].keys())
    g0s = np.array(g0s, float)

    mae_mean, mae_std = [], []
    r2_all = []
    fold_mae_map = {}

    for g0 in g0s:
        ytrue, yhat = reg_cache_all[family][float(g0)]
        fm = reg_fold_mae(ytrue, yhat)
        fold_mae_map[float(g0)] = fm
        mae_mean.append(float(np.mean(fm)))
        mae_std.append(float(np.std(fm)))
        r2_all.append(float(r2_score(ytrue, yhat)))

    mae_mean = np.array(mae_mean, float)
    mae_std = np.array(mae_std, float)
    r2_all = np.array(r2_all, float)

    # 选择“最佳G0”（以MAE最小为准；更稳定）
    best_idx = int(np.argmin(mae_mean))
    g0_best = float(g0s[best_idx])

    # 2) 画图
    fig = plt.figure(figsize=(8.2, 7.0), dpi=FIG_DPI)
    gs = fig.add_gridspec(2, 2, hspace=0.35, wspace=0.30)

    # (a) 任务定义示意：展示多个G0下的目标曲线（只画Target）
    ax = fig.add_subplot(gs[0, 0])
    beautify_ax(ax); add_panel_label(ax, "a")
    G_grid = np.sort(np.unique(G))
    # 取三个代表G0：低/中/高
    g0_show = [float(np.quantile(g0s, q)) for q in (0.15, 0.50, 0.85)]
    for g0 in g0_show:
        y = reg_target_on_grid(family, g0, G_grid)
        ax.plot(G_grid, y, label=f"Target (G0={g0:g})")
    ax.set_title(f"{family}: task definition")
    ax.set_xlabel("Glucose concentration (G)")
    ax.set_ylabel("Target output")
    ax.set_ylim(-0.05, 1.05)
    ax.legend(loc="best")

    # (b) 代表性demo：Target曲线 + 预测点（LOCO OOF）
    ax = fig.add_subplot(gs[0, 1])
    beautify_ax(ax); add_panel_label(ax, "b")
    g0_demo = float(G0_demo)
    # 若demo不在扫描列表中，回退到最接近的一个
    if float(g0_demo) not in reg_cache_all[family]:
        g0_demo = float(g0s[np.argmin(np.abs(g0s - g0_demo))])
    ytrue, yhat = reg_cache_all[family][float(g0_demo)]
    ord_idx = np.argsort(G)
    ax.plot(G[ord_idx], ytrue[ord_idx], label="Target")
    ax.scatter(G[ord_idx], yhat[ord_idx], s=32, label="Prediction (LOCO OOF)")
    ax.axvline(g0_demo, linestyle="--", linewidth=1.6)
    ax.set_title(f"{family} demo (G0={g0_demo:g})")
    ax.set_xlabel("Glucose concentration (G)")
    ax.set_ylabel("Output")
    ax.set_ylim(-0.05, 1.05)
    ax.legend(loc="best")

    # (c) 性能谱：MAE vs G0（误差带：外层折std）
    ax = fig.add_subplot(gs[1, 0])
    beautify_ax(ax); add_panel_label(ax, "c")
    ax.plot(g0s, mae_mean, marker="o", label="MAE (mean over LOCO folds)")
    ax.fill_between(g0s, mae_mean - mae_std, mae_mean + mae_std, alpha=0.2, label="±1 std")
    ax.axvline(g0_best, linestyle="--", linewidth=1.4)
    ax.set_title(f"{family} performance spectrum (outer LOCO)")
    ax.set_xlabel("G0")
    ax.set_ylabel("MAE (lower is better)")
    ax.legend(loc="best")

    # (d) 稳健性：最佳G0下，各外层折MAE箱线图
    ax = fig.add_subplot(gs[1, 1])
    beautify_ax(ax); add_panel_label(ax, "d")
    fm_best = fold_mae_map[g0_best]
    ax.boxplot([fm_best], labels=[f"G0={g0_best:g}"], showfliers=False)
    ax.set_title(f"{family} robustness across LOCO folds")
    ax.set_ylabel("MAE per fold")
    ax.set_xlabel("Best G0 (by MAE)")

    fn = OUTDIR / f"V43_panel_reg_{family}.png"
    fig.savefig(fn, bbox_inches="tight")
    plt.close(fig)
    display(Image(filename=str(fn)))

    # 返回用于全局记分板的摘要
    return dict(
        family=family,
        g0_best=g0_best,
        mae_best=float(mae_mean[best_idx]),
        mae_mean=float(np.mean(mae_mean)),
        r2_mean=float(np.mean(r2_all))
    )

reg_panel_rows = []
for fam in ["Notch", "DoubleTuning", "PiecewiseSat"]:
    reg_panel_rows.append(plot_reg_panel(fam, G0_demo=25.0))

reg_panel_df = pd.DataFrame(reg_panel_rows)
reg_panel_df.to_csv(OUTDIR / "regression_panels_summary_V43.csv", index=False)
display(reg_panel_df)

# -------------------------
# 13.2 固定二分类任务：Stripes / SineSign（2×2 面板）
# -------------------------
def binary_fold_accuracy(y_true, y_pred):
    """按外层折计算 accuracy 列表（单类折也可用，不退化）。"""
    accs = []
    for tr, te in outer_splits:
        accs.append(float(np.mean(y_true[te] == y_pred[te])))
    return np.array(accs, float)

def plot_fixed_binary_panel(tasks):
    """固定二分类任务的 2×2 面板：
    a: 任务定义（Target vs G）
    b: 代表性OOF（Target vs Prediction）
    c: 总体指标柱状图（phi / bal_acc）
    d: 外层折accuracy热图（每任务一行）
    """
    fig = plt.figure(figsize=(8.2, 7.0), dpi=FIG_DPI)
    gs = fig.add_gridspec(2, 2, hspace=0.35, wspace=0.30)

    # (a) 任务定义：Target随G变化（每任务一条）
    ax = fig.add_subplot(gs[0, 0])
    beautify_ax(ax); add_panel_label(ax, "a")
    ord_idx = np.argsort(G)
    for name, yb in tasks:
        ax.scatter(G[ord_idx], yb[ord_idx], s=22, label=f"Target ({name})")
    ax.set_title("Fixed binary tasks: task definition")
    ax.set_xlabel("Glucose concentration (G)")
    ax.set_ylabel("Class (0/1)")
    ax.legend(loc="best")

    # (b) 代表性OOF：用第一个任务示例展示（避免拥挤）
    ax = fig.add_subplot(gs[0, 1])
    beautify_ax(ax); add_panel_label(ax, "b")
    name0, yb0 = tasks[0]
    oof0 = cls_oof_cache[name0]
    ax.scatter(G[ord_idx], yb0[ord_idx], s=22, label="Target")
    ax.scatter(G[ord_idx], oof0[ord_idx], s=22, label="Prediction (LOCO OOF)")
    ax.set_title(f"OOF demo ({name0})")
    ax.set_xlabel("Glucose concentration (G)")
    ax.set_ylabel("Class (0/1)")
    ax.legend(loc="best")

    # (c) 总体指标：phi / bal_acc
    ax = fig.add_subplot(gs[1, 0])
    beautify_ax(ax); add_panel_label(ax, "c")
    names = [t[0] for t in tasks]
    phi = [float(cls_df.loc[cls_df["task"]==nm, "phi_acc"].values[0]) for nm in names]
    bal = [float(cls_df.loc[cls_df["task"]==nm, "bal_acc"].values[0]) for nm in names]
    x_pos = np.arange(len(names))
    ax.bar(x_pos - 0.18, phi, width=0.36, label="Phi-accuracy")
    ax.bar(x_pos + 0.18, bal, width=0.36, label="Balanced accuracy")
    ax.axhline(0.5, linestyle="--", linewidth=1.0)
    ax.set_xticks(x_pos)
    ax.set_xticklabels(names, rotation=10)
    ax.set_ylim(0.0, 1.02)
    ax.set_title("Overall metrics (outer LOCO OOF)")
    ax.set_ylabel("Score (higher is better)")
    ax.legend(loc="best")

    # (d) 外层折accuracy热图（每任务一行）
    ax = fig.add_subplot(gs[1, 1])
    beautify_ax(ax); add_panel_label(ax, "d")
    acc_mat = []
    for nm, yb in tasks:
        oof = cls_oof_cache[nm]
        acc_mat.append(binary_fold_accuracy(yb, oof))
    acc_mat = np.vstack(acc_mat)
    im = ax.imshow(acc_mat, aspect="auto", vmin=0.0, vmax=1.0, cmap="turbo")
    ax.set_yticks(np.arange(len(names)))
    ax.set_yticklabels(names)
    # x轴刻度避免拥挤：最多标 8 个刻度
    tick_id = np.linspace(0, n_outer-1, min(8, n_outer)).astype(int)
    ax.set_xticks(tick_id)
    ax.set_xticklabels([str(i) for i in tick_id])
    ax.set_xlabel("Outer LOCO fold")
    ax.set_ylabel("Task")
    ax.set_title("Fold-wise accuracy heatmap")
    cb = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    cb.set_label("Accuracy")

    fn = OUTDIR / "V43_panel_binary_fixed.png"
    fig.savefig(fn, bbox_inches="tight")
    plt.close(fig)
    display(Image(filename=str(fn)))

plot_fixed_binary_panel([
    ("Stripes_m4", stripes_target(m=4)),
    ("SineSign_m2", sine_sign_target(m=2)),
])

# -------------------------
# 13.3 Band-pass（扫G0）：2×2 面板
# -------------------------
def plot_bandpass_panel(G0_demo=25.0):
    # 从 band_cache 读取所有G0对应的 (ytrue, ypred, w)
    g0s = np.array(sorted(band_cache.keys()), float)

    # 逐G0总体指标（全体OOF拼接计算）
    phi_list, bal_list = [], []
    for g0 in g0s:
        ytrue, ypred, _ = band_cache[float(g0)]
        met = binary_metrics(ytrue, ypred)
        phi_list.append(met["phi_acc"])
        bal_list.append(met["bal_acc"])
    phi_list = np.array(phi_list, float)
    bal_list = np.array(bal_list, float)

    # 逐折accuracy热图：每格为该fold在该G0下的accuracy（不退化）
    acc_mat = np.zeros((n_outer, len(g0s)), float)
    for j, g0 in enumerate(g0s):
        ytrue, ypred, _ = band_cache[float(g0)]
        acc_mat[:, j] = binary_fold_accuracy(ytrue, ypred)

    # 选择最佳G0（以phi最大；若并列取bal更大）
    best_idx = int(np.argmax(phi_list + 1e-6 * bal_list))
    g0_best = float(g0s[best_idx])

    # demo G0
    g0_demo = float(G0_demo)
    if float(g0_demo) not in band_cache:
        g0_demo = float(g0s[np.argmin(np.abs(g0s - g0_demo))])
    ytrue_demo, ypred_demo, w_demo = band_cache[float(g0_demo)]
    w_demo = float(np.asarray(w_demo).ravel()[0])

    fig = plt.figure(figsize=(8.2, 7.0), dpi=FIG_DPI)
    gs = fig.add_gridspec(2, 2, hspace=0.35, wspace=0.30)

    # (a) 任务定义示意：band window（只画Target）
    ax = fig.add_subplot(gs[0, 0])
    beautify_ax(ax); add_panel_label(ax, "a")
    ord_idx = np.argsort(G)
    ax.scatter(G[ord_idx], ytrue_demo[ord_idx], s=22, label="Target")
    ax.axvline(g0_demo, linestyle="--", linewidth=1.6, label=f"G0={g0_demo:g}")
    ax.set_title(f"Band-pass: task definition (w={w_demo:g})")
    ax.set_xlabel("Glucose concentration (G)")
    ax.set_ylabel("Class (0/1)")
    ax.legend(loc="best")

    # (b) demo：Target vs Prediction（只用点）
    ax = fig.add_subplot(gs[0, 1])
    beautify_ax(ax); add_panel_label(ax, "b")
    ax.scatter(G[ord_idx], ytrue_demo[ord_idx], s=22, label="Target")
    ax.scatter(G[ord_idx], ypred_demo[ord_idx], s=22, label="Prediction (LOCO OOF)")
    ax.axvline(g0_demo, linestyle="--", linewidth=1.6)
    ax.set_title(f"Band-pass demo (G0={g0_demo:g})")
    ax.set_xlabel("Glucose concentration (G)")
    ax.set_ylabel("Class (0/1)")
    ax.legend(loc="best")

    # (c) 性能谱：phi/bal vs G0（并标记最佳）
    ax = fig.add_subplot(gs[1, 0])
    beautify_ax(ax); add_panel_label(ax, "c")
    ax.plot(g0s, phi_list, marker="o", label="Phi-accuracy")
    ax.plot(g0s, bal_list, marker="o", label="Balanced accuracy")
    ax.axvline(g0_best, linestyle="--", linewidth=1.4, label=f"Best G0={g0_best:g}")
    ax.set_ylim(0.0, 1.02)
    ax.set_title("Band-pass performance spectrum (outer LOCO OOF)")
    ax.set_xlabel("G0")
    ax.set_ylabel("Score (higher is better)")
    ax.legend(loc="best")

    # (d) 稳健性：fold-wise accuracy heatmap（fold × G0）
    ax = fig.add_subplot(gs[1, 1])
    beautify_ax(ax); add_panel_label(ax, "d")
    im = ax.imshow(acc_mat, aspect="auto", vmin=0.0, vmax=1.0, cmap="turbo")
    ax.set_title("Fold-wise accuracy heatmap")
    ax.set_xlabel("G0")
    ax.set_ylabel("Outer LOCO fold")
    # x轴标注少量G0刻度避免拥挤
    tick_id = np.linspace(0, len(g0s)-1, min(6, len(g0s))).astype(int)
    ax.set_xticks(tick_id)
    ax.set_xticklabels([f"{g0s[i]:g}" for i in tick_id], rotation=0)
    cb = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    cb.set_label("Accuracy")

    fn = OUTDIR / "V43_panel_bandpass.png"
    fig.savefig(fn, bbox_inches="tight")
    plt.close(fig)
    display(Image(filename=str(fn)))

    # 保存数值（便于写文稿与复核）
    pd.DataFrame(dict(G0=g0s, phi_acc=phi_list, bal_acc=bal_list)).to_csv(
        OUTDIR / "bandpass_spectrum_V43.csv", index=False
    )

plot_bandpass_panel(G0_demo=25.0)

# -------------------------
# 13.4 多级任务（Quantization / Ordinal）：2×2 面板 + 折间波动
# -------------------------
def fold_macro_f1(y_true, y_pred):
    """按外层折计算macro_f1列表（小测试集可能波动较大，用于“稳健性”参考）。"""
    f1s = []
    for tr, te in outer_splits:
        f1s.append(float(f1_score(y_true[te], y_pred[te], average="macro")))
    return np.array(f1s, float)

def plot_multilevel_panel():
    fig = plt.figure(figsize=(8.2, 7.0), dpi=FIG_DPI)
    gs = fig.add_gridspec(2, 2, hspace=0.35, wspace=0.30)

    # (a) 任务定义：量化边界（把edges从归一化坐标映射回G）
    ax = fig.add_subplot(gs[0, 0])
    beautify_ax(ax); add_panel_label(ax, "a")
    ax.hist(G, bins=10)
    G_edges = G_min + np.asarray(edges, float) * (G_max - G_min)
    for e in G_edges:
        ax.axvline(float(e), linestyle="--", linewidth=1.2)
    ax.set_title("Quantization: bin edges (K=4)")
    ax.set_xlabel("Glucose concentration (G)")
    ax.set_ylabel("Count")

    # (b) Quantization confusion
    ax = fig.add_subplot(gs[0, 1])
    beautify_ax(ax); add_panel_label(ax, "b")
    im = ax.imshow(cm_q, cmap="viridis")
    ax.set_title("Quantization confusion (K=4)")
    ax.set_xlabel("Pred")
    ax.set_ylabel("True")
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

    # (c) 指标柱状图（Quantization vs Ordinal）
    ax = fig.add_subplot(gs[1, 0])
    beautify_ax(ax); add_panel_label(ax, "c")
    labels = ["Quant macro_f1", "Quant bal_acc", "Ord macro_f1", "Ord bal_acc", "Ord mae_level"]
    vals = [
        float(met_q["macro_f1"]), float(met_q["bal_acc"]),
        float(met_ord["macro_f1"]), float(met_ord["bal_acc"]), float(met_ord["mae_level"])
    ]
    ax.bar(np.arange(len(labels)), vals)
    ax.set_xticks(np.arange(len(labels)))
    ax.set_xticklabels(labels, rotation=15, ha="right")
    ax.set_ylim(0.0, 1.05)
    ax.set_title("Multilevel metrics (outer LOCO OOF)")
    ax.set_ylabel("Score (higher is better)")

    # (d) 稳健性：折间macro_f1分布（Quant与Ordinal对比）
    ax = fig.add_subplot(gs[1, 1])
    beautify_ax(ax); add_panel_label(ax, "d")
    f1_q = fold_macro_f1(yq, oof_q)
    f1_o = fold_macro_f1(y_ord_true, y_ord_pred)
    ax.boxplot([f1_q, f1_o], labels=["Quant", "Ordinal"], showfliers=False)
    ax.set_title("Robustness across LOCO folds")
    ax.set_ylabel("macro_f1 per fold")

    fn = OUTDIR / "V43_panel_multilevel.png"
    fig.savefig(fn, bbox_inches="tight")
    plt.close(fig)
    display(Image(filename=str(fn)))

plot_multilevel_panel()

# -------------------------
# 13.5 Ranking：2×2 面板（任务定义 + 分布 + 稳健性 + 汇总）
# -------------------------
def plot_ranking_panel():
    fig = plt.figure(figsize=(8.2, 7.0), dpi=FIG_DPI)
    gs = fig.add_gridspec(2, 2, hspace=0.35, wspace=0.30)

    # (a) 任务定义：用浓度排序示意（不画真实配对，避免拥挤）
    ax = fig.add_subplot(gs[0, 0])
    beautify_ax(ax); add_panel_label(ax, "a")
    ord_idx = np.argsort(G)
    ax.plot(np.arange(len(G)), G[ord_idx], marker="o")
    ax.set_title("Ranking: task definition (order by G)")
    ax.set_xlabel("Sample index (sorted)")
    ax.set_ylabel("Glucose concentration (G)")

    # (b) AUC分布直方图
    ax = fig.add_subplot(gs[0, 1])
    beautify_ax(ax); add_panel_label(ax, "b")
    if len(auc_folds):
        ax.hist(auc_folds, bins=10)
    else:
        ax.text(0.5, 0.5, "No valid folds", ha="center", va="center", transform=ax.transAxes)
    ax.set_title("Pairwise ranking AUC (outer LOCO)")
    ax.set_xlabel("AUC")
    ax.set_ylabel("Count")

    # (c) 折间AUC散点（稳健性）
    ax = fig.add_subplot(gs[1, 0])
    beautify_ax(ax); add_panel_label(ax, "c")
    if len(auc_folds):
        ax.scatter(np.arange(len(auc_folds)), auc_folds, s=28)
        ax.set_ylim(0.0, 1.02)
    ax.set_title("Fold-wise AUC")
    ax.set_xlabel("Fold (used)")
    ax.set_ylabel("AUC")

    # (d) 汇总条形图
    ax = fig.add_subplot(gs[1, 1])
    beautify_ax(ax); add_panel_label(ax, "d")
    ax.bar(["AUC mean", "ACC mean"], [rank_metrics["auc_mean"], rank_metrics["acc_mean"]])
    ax.set_ylim(0.0, 1.02)
    ax.set_title(f"Ranking summary (folds used={rank_metrics['n_folds_used']})")
    ax.set_ylabel("Score")

    fn = OUTDIR / "V43_panel_ranking.png"
    fig.savefig(fn, bbox_inches="tight")
    plt.close(fig)
    display(Image(filename=str(fn)))

plot_ranking_panel()

# -------------------------
# 13.6 全任务记分板（建议论文主图：表格 + 导出csv）
# -------------------------
# 说明：
# - 不同任务指标量纲不同，强行做同一色标热图会误导。
# - 更通用且可复用的做法：表格式记分板（每行一个任务，每列一个主要指标）。
score_rows = []

# 回归：用 mean R2 across G0（越大越好）与 mean MAE across G0（越小越好）
for _, r in reg_panel_df.iterrows():
    score_rows.append(dict(
        task=f"Reg_{r['family']}",
        metric_main="R2_mean_across_G0",
        value=float(r["r2_mean"]),
        metric_aux="MAE_mean_across_G0",
        aux_value=float(r["mae_mean"]),
    ))

# 固定二分类：phi_acc / bal_acc
for nm in cls_df["task"].tolist():
    row = cls_df.loc[cls_df["task"]==nm].iloc[0]
    score_rows.append(dict(
        task=f"Bin_{nm}",
        metric_main="Phi-accuracy",
        value=float(row["phi_acc"]),
        metric_aux="Balanced accuracy",
        aux_value=float(row["bal_acc"]),
    ))

# Band-pass：取最优G0下phi_acc与bal_acc
band_spectrum = pd.read_csv(OUTDIR / "bandpass_spectrum_V43.csv")
best_id = int(np.argmax(band_spectrum["phi_acc"].values + 1e-6 * band_spectrum["bal_acc"].values))
score_rows.append(dict(
    task="Bin_BandPass(best G0)",
    metric_main="Phi-accuracy",
    value=float(band_spectrum["phi_acc"].values[best_id]),
    metric_aux="Balanced accuracy",
    aux_value=float(band_spectrum["bal_acc"].values[best_id]),
))

# 多级：Quant与Ordinal
score_rows.append(dict(
    task="Multi_Quant(K=4)",
    metric_main="macro_f1",
    value=float(met_q["macro_f1"]),
    metric_aux="bal_acc",
    aux_value=float(met_q["bal_acc"]),
))
score_rows.append(dict(
    task="Multi_Ordinal",
    metric_main="macro_f1",
    value=float(met_ord["macro_f1"]),
    metric_aux="mae_level (lower better)",
    aux_value=float(met_ord["mae_level"]),
))

# 排序
score_rows.append(dict(
    task="Rank_Pairwise",
    metric_main="AUC_mean",
    value=float(rank_metrics["auc_mean"]),
    metric_aux="ACC_mean",
    aux_value=float(rank_metrics["acc_mean"]),
))

score_df = pd.DataFrame(score_rows)
score_df.to_csv(OUTDIR / "scoreboard_V43.csv", index=False)
display(score_df)

# 生成“表格图”（用于论文/汇报直接贴图）
fig = plt.figure(figsize=(10.6, 0.42*len(score_df)+1.2), dpi=FIG_DPI)
ax = fig.add_subplot(111)
ax.axis("off")
col_labels = ["Task", "Main metric", "Value", "Aux metric", "Aux value"]
cell_text = []
for _, r in score_df.iterrows():
    cell_text.append([
        str(r["task"]),
        str(r["metric_main"]),
        f"{float(r['value']):.4f}",
        str(r["metric_aux"]),
        f"{float(r['aux_value']):.4f}",
    ])
tbl = ax.table(cellText=cell_text, colLabels=col_labels, cellLoc="left", colLoc="left", loc="center")
tbl.auto_set_font_size(False)
tbl.set_fontsize(9)
tbl.scale(1.0, 1.2)

fn = OUTDIR / "V43_scoreboard.png"
fig.savefig(fn, bbox_inches="tight")
plt.close(fig)
display(Image(filename=str(fn)))

print("Saved universal panels + scoreboard (V43).")


In [ ]:

# ============================================================
# 14. 自检（关键输出是否存在，V43）
# ============================================================
required = [
    OUTDIR / "regression_curves_V43.csv",
    OUTDIR / "classification_scores_V43.csv",
    OUTDIR / "bandpass_curve_V43.csv",
    OUTDIR / "quantization_metrics_V43.csv",
    OUTDIR / "ordinal_metrics_V43.csv",
    OUTDIR / "pairwise_ranking_metrics_V43.csv",
    OUTDIR / "feature_subset_capacity_V43.csv",

    # 旧版单图（仍会生成）
    OUTDIR / "V43_ranking.png",
    OUTDIR / "V43_feature_capacity.png",

    # 新版通用面板与记分板
    OUTDIR / "V43_panel_reg_Notch.png",
    OUTDIR / "V43_panel_reg_DoubleTuning.png",
    OUTDIR / "V43_panel_reg_PiecewiseSat.png",
    OUTDIR / "V43_panel_binary_fixed.png",
    OUTDIR / "V43_panel_bandpass.png",
    OUTDIR / "V43_panel_multilevel.png",
    OUTDIR / "V43_panel_ranking.png",
    OUTDIR / "V43_scoreboard.png",
    OUTDIR / "scoreboard_V43.csv",
]
missing = [str(p) for p in required if not p.exists()]
if missing:
    raise FileNotFoundError("Missing outputs:\n" + "\n".join(missing))
print("Self-check passed. Key outputs exist (V43).")
